<a href="https://colab.research.google.com/github/WangariLM/telecom_churn_prediction/blob/main/notebooks/01_setup_and_first_look.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:

# =============================================================================
# MASTER SETUP CELL — RUN THIS FIRST EVERY SINGLE SESSION
# Mounts Drive, sets paths, recreates variables
# Heavy computation like GridSearchCV only runs if no saved model exists
# =============================================================================

# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Set up paths
import sys
import os
import joblib
import importlib

#import evaluate
#importlib.reload(evaluate)
PROJECT_PATH = '/content/drive/MyDrive/telecom_churn_prediction'
SRC_PATH = os.path.join(PROJECT_PATH, 'src')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Step 3: Import all modules
import config
import data_loader
import feature_engineering
import preprocessing
import train
import evaluate
importlib.reload(evaluate)
importlib.reload(config)
importlib.reload(data_loader)
importlib.reload(feature_engineering)
importlib.reload(preprocessing)
importlib.reload(train)


# Step 4: Recreate data variables
df_clean = data_loader.load_and_clean_data()

# Step 5: Recreate preprocessing variables
X_train_processed, X_test_processed, y_train, y_test, pipeline = \
    preprocessing.run_preprocessing_pipeline(df_clean)

# Step 6: Load model from disk if it exists
# If it does not exist run training once and save it
MODEL_PATH = config.MODEL_PATH


if os.path.exists(MODEL_PATH):
    # Model already trained and saved — just load it
    best_model = joblib.load(MODEL_PATH)
    grid_search = None
    print("Model loaded from disk — skipping GridSearchCV")
else:
    # First time only — run GridSearchCV and save
    print("No saved model found — running GridSearchCV for the first time")
    print("This will take a few minutes...")
    best_model, grid_search = train.run_training_pipeline(
        X_train_processed,
        X_test_processed,
        y_train,
        y_test,
        pipeline
    )
    print("Model trained and saved — future sessions will load from disk")


metrics = evaluate.run_evaluation_pipeline(
    model=best_model,
    X_test_processed=X_test_processed,
    y_test=y_test,
    preprocessing_pipeline=pipeline,
    customer_index=0
)

# Load full prediction pipeline
full_pipeline = joblib.load(config.FULL_PIPELINE_PATH)
print(f"Full pipeline loaded: {[s[0] for s in full_pipeline.steps]}")

# Step 7: Confirm everything is ready
print("\n" + "=" * 50)
print("SESSION READY!")
print("=" * 50)
print(f"df_clean          : {df_clean.shape}")
print(f"X_train_processed : {X_train_processed.shape}")
print(f"X_test_processed  : {X_test_processed.shape}")
print(f"y_train           : {y_train.shape}")
print(f"y_test            : {y_test.shape}")
print(f"Model loaded      : {type(best_model).__name__}")
print("=" * 50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded from disk — skipping GridSearchCV

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Stayed (0)       0.89      0.75      0.82      1033
 Churned (1)       0.52      0.75      0.62       372

    accuracy                           0.75      1405
   macro avg       0.71      0.75      0.72      1405
weighted avg       0.80      0.75      0.76      1405



/usr/local/lib/python3.12/dist-packages/shap/explainers/_linear.py:123: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)


Full pipeline loaded: ['preprocessing', 'logisticregression']

SESSION READY!
df_clean          : (7021, 20)
X_train_processed : (5616, 40)
X_test_processed  : (1405, 40)
y_train           : (5616,)
y_test            : (1405,)
Model loaded      : Pipeline


Chapter 1
.Creating all the necessary folders
. Unzipping the dataset drom zip file into csv file and renaming it
. First Look into the dataset

In [ ]:

import os

# Define your project path
PROJECT_PATH = '/content/drive/MyDrive/telecom_churn_prediction'

# Check the data folder
data_path = os.path.join(PROJECT_PATH, 'data')
print("Files in data folder:")
for file in os.listdir(data_path):
    print(f"  {file}")

Files in data folder:
  Telco-Customer-Churn.zip
  telco_churn.csv


In [ ]:

# Define all folders needed
folders = [
    'data',
    'notebooks',
    'src',
    'models',
    'reports/figures',
    'tests'
]

# Create each folder
for folder in folders:
    folder_path = os.path.join(PROJECT_PATH, folder)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created: {folder_path}")

print("\nProject structure created successfully!")

Created: /content/drive/MyDrive/telecom_churn_prediction/data
Created: /content/drive/MyDrive/telecom_churn_prediction/notebooks
Created: /content/drive/MyDrive/telecom_churn_prediction/src
Created: /content/drive/MyDrive/telecom_churn_prediction/models
Created: /content/drive/MyDrive/telecom_churn_prediction/reports/figures
Created: /content/drive/MyDrive/telecom_churn_prediction/tests

Project structure created successfully!


In [ ]:

import os

PROJECT_PATH = '/content/drive/MyDrive/telecom_churn_prediction'

print("Folders found:")
for item in sorted(os.listdir(PROJECT_PATH)):
    print(f"  {item}")

Folders found:
  data
  models
  notebooks
  reports
  src
  tests


In [ ]:
# Extracting the dataset from zip file
import zipfile

zip_path = os.path.join(PROJECT_PATH, 'data', 'Telco-Customer-Churn.zip')
extract_path = os.path.join(PROJECT_PATH, 'data')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
    print("Files extracted:")
    for file in zip_ref.namelist():
        print(f"  {file}")

Files extracted:
  WA_Fn-UseC_-Telco-Customer-Churn.csv


In [ ]:
#Renaming the csv file
import shutil

# Find the extracted CSV
old_name = os.path.join(PROJECT_PATH, 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
new_name = os.path.join(PROJECT_PATH, 'data', 'telco_churn.csv')

# Rename it
if os.path.exists(old_name):
    shutil.move(old_name, new_name)
    print("File renamed successfully to telco_churn.csv")
else:
    print("File already renamed or not found")
    print("Files in data folder:")
    for file in os.listdir(data_path):
        print(f"  {file}")

File renamed successfully to telco_churn.csv


In [ ]:
#Verifying if everything is in order
def show_project_structure(path, indent=0):
    """Display project folder structure"""
    for item in sorted(os.listdir(path)):
        item_path = os.path.join(path, item)
        print(' ' * indent + '|-- ' + item)
        if os.path.isdir(item_path):
            show_project_structure(item_path, indent + 4)

print("telecom-churn-prediction/")
show_project_structure(PROJECT_PATH)

telecom-churn-prediction/
|-- data
    |-- Telco-Customer-Churn.zip
    |-- telco_churn.csv
|-- models
|-- notebooks
|-- reports
    |-- figures
|-- src
|-- tests


In [ ]:
#A look into the data
import pandas as pd

# Load the data
df = pd.read_csv(new_name)

# Basic information
print("Shape of dataset:")
print(f"  {df.shape[0]} rows (customers)")
print(f"  {df.shape[1]} columns (features)")

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names and data types:")
print(df.dtypes)

print("\nBasic statistics:")
display(df.describe())

Shape of dataset:
  7043 rows (customers)
  21 columns (features)

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Column names and data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Basic statistics:


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Configuration

In [ ]:


# =============================================================================
# CHAPTER 2: Configuration
# Writes config.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================
config_content = '''
# =============================================================================
# config.py
# Central configuration file for the telecom churn prediction pipeline.
# All constants, paths, and settings are defined here.
# Never hardcode these values anywhere else in the codebase.
# =============================================================================

import os

# =============================================================================
# PROJECT PATHS
# =============================================================================

# Base project path — all other paths are relative to this
BASE_DIR = '/content/drive/MyDrive/telecom_churn_prediction'

# Data paths
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DATA_PATH = os.path.join(DATA_DIR, 'telco_churn.csv')

# Model output path
MODEL_DIR = os.path.join(BASE_DIR, 'models')
MODEL_PATH = os.path.join(MODEL_DIR, 'logistic_regression_pipeline.pkl')

# Reports and figures path
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')
FIGURES_DIR = os.path.join(REPORTS_DIR, 'figures')

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

# Fixed random seed used everywhere — ensures identical results on every run
RANDOM_SEED = 42

# =============================================================================
# DATA SETTINGS
# =============================================================================

# The column we are predicting
TARGET_COLUMN = 'Churn'

# Column to drop — just an identifier, no predictive value
DROP_COLUMNS = ['customerID']

# Numerical features in the raw dataset
NUMERICAL_FEATURES = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

# Binary categorical features — only two possible values (Yes/No)
BINARY_FEATURES = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

# Multi-class categorical features — more than two possible values
MULTICLASS_FEATURES = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

# SeniorCitizen is stored as int (0/1) but is categorical in nature
# We handle it separately from other numerical features
SENIOR_CITIZEN_FEATURE = 'SeniorCitizen'

# =============================================================================
# ENGINEERED FEATURE NAMES
# =============================================================================

# These are the new columns our feature engineering transformer will create
ENGINEERED_NUMERICAL_FEATURES = [
    'TotalServices',
    'SpendPerService',
    'ChargesRatio',
    'ContractRiskScore',
    'TenureContractInteraction'
]

ENGINEERED_BINARY_FEATURES = [
    'HasPremiumServices',
    'IsAutomatedPayment'
]

ENGINEERED_CATEGORICAL_FEATURES = [
    'TenureGroup'
]

# =============================================================================
# DATA SPLIT SETTINGS
# =============================================================================

# Proportion of data reserved for final testing — never touched during training
TEST_SIZE = 0.2

# Proportion of training data used for validation during development
VALIDATION_SIZE = 0.2

# =============================================================================
# CROSS VALIDATION SETTINGS
# =============================================================================

# Number of folds for stratified k-fold cross validation
# 5 is the production standard — balances reliability and computation time
CV_FOLDS = 5

# Primary metric used to select the best model during cross validation
# roc_auc is robust to class imbalance and measures overall discrimination
SCORING_METRIC = 'roc_auc'

# =============================================================================
# HYPERPARAMETER TUNING GRID
# =============================================================================

# C is the inverse of regularization strength (C = 1/lambda)
# Small C = strong regularization = simpler model
# Large C = weak regularization = more complex model
# We search across several orders of magnitude to find the sweet spot

HYPERPARAMETER_GRID = {
    # C values to try — covering a wide range from strong to weak regularization
    'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100],

    # Penalty type — L1 can zero out features, L2 shrinks them
    'logisticregression__penalty': ['l1', 'l2'],

    # Solver must be compatible with penalty type
    # liblinear: works with both L1 and L2, good for small datasets
    # saga: works with both L1 and L2, better for large datasets
    'logisticregression__solver': ['liblinear', 'saga']
}

# =============================================================================
# MODEL TRAINING SETTINGS
# =============================================================================

# Maximum iterations for the logistic regression solver to converge
# 1000 is generous — prevents convergence warnings on complex data
MAX_ITER = 1000

# Class weight setting
# balanced: automatically adjusts weights inversely proportional to class frequency
# This is a second line of defence against class imbalance alongside SMOTE
CLASS_WEIGHT = 'balanced'

# =============================================================================
# SMOTE SETTINGS
# =============================================================================

# Sampling strategy for SMOTE
# auto: resamples minority class to match majority class exactly
SMOTE_SAMPLING_STRATEGY = 'auto'

# Number of nearest neighbours SMOTE uses to generate synthetic samples
# 5 is the default and works well in most cases
SMOTE_K_NEIGHBOURS = 5

# =============================================================================
# EVALUATION SETTINGS
# =============================================================================

# Default classification threshold
# We expose this as a config value because the business may want to adjust it
# Lower threshold = higher recall = catch more churners but more false alarms
# Higher threshold = higher precision = fewer false alarms but miss more churners
CLASSIFICATION_THRESHOLD = 0.5

# =============================================================================
# LOGGING SETTINGS
# =============================================================================

# Log level — INFO shows progress, DEBUG shows everything
LOG_LEVEL = 'INFO'

# Log format — timestamp, module name, level, message
LOG_FORMAT = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'

# =============================================================================
# FIGURE SETTINGS
# =============================================================================

# DPI for saved figures — 300 is publication quality
FIGURE_DPI = 300

# Default figure size
FIGURE_SIZE = (10, 6)
'''

# Write config.py to the src folder
config_path = os.path.join(PROJECT_PATH, 'src', 'config.py')

with open(config_path, 'w') as f:
    f.write(config_content)

print(f"config.py written successfully to:")
print(f"  {config_path}")

config.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/config.py


In [ ]:

# Read and display the file we just wrote
with open(config_path, 'r') as f:
    print(f.read())


# =============================================================================
# config.py
# Central configuration file for the telecom churn prediction pipeline.
# All constants, paths, and settings are defined here.
# Never hardcode these values anywhere else in the codebase.
# =============================================================================

import os

# =============================================================================
# PROJECT PATHS
# =============================================================================

# Base project path — all other paths are relative to this
BASE_DIR = '/content/drive/MyDrive/telecom_churn_prediction'

# Data paths
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DATA_PATH = os.path.join(DATA_DIR, 'telco_churn.csv')

# Model output path
MODEL_DIR = os.path.join(BASE_DIR, 'models')
MODEL_PATH = os.path.join(MODEL_DIR, 'logistic_regression_pipeline.pkl')

# Reports and figures path
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')
F

In [ ]:

import sys

# Add src folder to Python path so we can import from it
sys.path.append(os.path.join(PROJECT_PATH, 'src'))

# Import config
import config

# Verify a few values
print("Verifying config values:")
print(f"  TARGET_COLUMN: {config.TARGET_COLUMN}")
print(f"  TEST_SIZE: {config.TEST_SIZE}")
print(f"  CV_FOLDS: {config.CV_FOLDS}")
print(f"  RANDOM_SEED: {config.RANDOM_SEED}")
print(f"  NUMERICAL_FEATURES: {config.NUMERICAL_FEATURES}")
print(f"  SCORING_METRIC: {config.SCORING_METRIC}")
print(f"\nconfig.py imported successfully!")

Verifying config values:
  TARGET_COLUMN: Churn
  TEST_SIZE: 0.2
  CV_FOLDS: 5
  RANDOM_SEED: 42
  NUMERICAL_FEATURES: ['tenure', 'MonthlyCharges', 'TotalCharges']
  SCORING_METRIC: roc_auc

config.py imported successfully!


Creating the data_loader.py

This file is responsible for one thing only ie loading the raw CSV and returning a clean dataframe ready for feature engineering. It does not do any feature engineering or preprocessing. Just loading and basic cleaning.
Specifically it will:
1. Load the raw CSV file
2. Drop customerID
3. Fix TotalCharges from string to float
4. Convert Churn from Yes/No to 1/0
5. Log every step so we can see exactly what is happening

In [ ]:

# =============================================================================
# CHAPTER 3: DATA LOADER
# Writes data_loader.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================
data_loader_content = '''
# =============================================================================
# data_loader.py
# Responsible for loading raw data and performing initial cleaning.
# No feature engineering happens here — just getting data into a clean state.
# =============================================================================

import logging
import pandas as pd
import numpy as np
import os
import sys

# Add src to path so we can import config
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config

# Set up logger for this module
logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)


def load_raw_data(filepath: str = config.RAW_DATA_PATH) -> pd.DataFrame:
    """
    Load raw CSV data from disk.

    Parameters
    ----------
    filepath : str
        Path to the raw CSV file. Defaults to path in config.

    Returns
    -------
    pd.DataFrame
        Raw dataframe exactly as it appears in the CSV.

    Raises
    ------
    FileNotFoundError
        If the CSV file does not exist at the given path.
    """
    logger.info(f"Loading raw data from: {filepath}")

    if not os.path.exists(filepath):
        logger.error(f"Data file not found at: {filepath}")
        raise FileNotFoundError(f"No file found at {filepath}")

    df = pd.read_csv(filepath)
    logger.info(f"Raw data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")

    return df


def drop_unnecessary_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop columns that carry no predictive value.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe.

    Returns
    -------
    pd.DataFrame
        Dataframe with unnecessary columns removed.
    """
    logger.info(f"Dropping columns: {config.DROP_COLUMNS}")

    # Only drop columns that actually exist in the dataframe
    cols_to_drop = [col for col in config.DROP_COLUMNS if col in df.columns]
    df = df.drop(columns=cols_to_drop)

    logger.info(f"Remaining columns: {df.shape[1]}")
    return df


def fix_total_charges(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert TotalCharges from string to float.

    TotalCharges is stored as object dtype because some rows contain
    blank spaces instead of numbers. These correspond to new customers
    with zero tenure who have never been billed. We replace blanks
    with 0 since that is the business correct value.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with TotalCharges as object dtype.

    Returns
    -------
    pd.DataFrame
        Dataframe with TotalCharges as float64.
    """
    logger.info("Fixing TotalCharges dtype from object to float")

    # Count blank spaces before fixing
    blank_count = df[df['TotalCharges'].str.strip() == '']['TotalCharges'].count()
    logger.info(f"Found {blank_count} blank TotalCharges values — replacing with 0")

    # Replace blank spaces with 0 then convert to float
    df['TotalCharges'] = df['TotalCharges'].str.strip()
    df['TotalCharges'] = df['TotalCharges'].replace('', '0')
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # Check if any NaN values remain after conversion
    remaining_nulls = df['TotalCharges'].isnull().sum()
    if remaining_nulls > 0:
        logger.warning(f"{remaining_nulls} NaN values remain in TotalCharges after conversion")
        logger.warning("Filling remaining NaN values with median")
        df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
    else:
        logger.info("TotalCharges conversion successful — no NaN values remaining")

    return df


def encode_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert target column Churn from Yes/No strings to binary 1/0.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with Churn as object dtype containing Yes/No.

    Returns
    -------
    pd.DataFrame
        Dataframe with Churn as int64 containing 1/0.
    """
    logger.info("Encoding target column: Yes -> 1, No -> 0")

    df[config.TARGET_COLUMN] = df[config.TARGET_COLUMN].map({'Yes': 1, 'No': 0})

    # Verify encoding worked correctly
    unique_values = df[config.TARGET_COLUMN].unique()
    null_count = df[config.TARGET_COLUMN].isnull().sum()

    logger.info(f"Target column unique values after encoding: {unique_values}")
    logger.info(f"Churn rate: {df[config.TARGET_COLUMN].mean():.2%}")

    if null_count > 0:
        logger.error(f"Found {null_count} NaN values in target after encoding")
        raise ValueError(f"Target encoding failed — {null_count} unexpected values found")

    return df


def validate_data(df: pd.DataFrame) -> None:
    """
    Run basic validation checks on the cleaned dataframe.
    Logs warnings for any issues found but does not stop the pipeline.

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned dataframe to validate.

    Returns
    -------
    None
    """
    logger.info("Running data validation checks")

    # Check 1: Expected number of rows
    if df.shape[0] < 7000:
        logger.warning(f"Fewer rows than expected: {df.shape[0]} (expected ~7043)")

    # Check 2: Check for missing values across all columns
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]

    if len(cols_with_nulls) > 0:
        logger.warning(f"Columns with missing values after cleaning:")
        for col, count in cols_with_nulls.items():
            logger.warning(f"  {col}: {count} missing values")
    else:
        logger.info("No missing values found after cleaning")

    # Check 3: Verify target column only contains 0 and 1
    valid_target_values = set(df[config.TARGET_COLUMN].unique())
    if not valid_target_values.issubset({0, 1}):
        logger.error(f"Target column contains unexpected values: {valid_target_values}")
    else:
        logger.info("Target column validation passed")

    # Check 4: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        logger.warning(f"Found {duplicate_count} duplicate rows")
    else:
        logger.info("No duplicate rows found")

    # Check 5: Verify TotalCharges is now numeric
    if df['TotalCharges'].dtype != 'float64':
        logger.error(f"TotalCharges dtype is {df['TotalCharges'].dtype} — expected float64")
    else:
        logger.info("TotalCharges dtype validation passed")

    logger.info("Data validation complete")


def load_and_clean_data(filepath: str = config.RAW_DATA_PATH) -> pd.DataFrame:
    """
    Master function that runs the complete data loading and cleaning pipeline.
    This is the main function called by other modules.

    Steps:
        1. Load raw CSV
        2. Drop unnecessary columns
        3. Fix TotalCharges dtype
        4. Encode target variable
        5. Validate cleaned data

    Parameters
    ----------
    filepath : str
        Path to raw CSV file. Defaults to path in config.

    Returns
    -------
    pd.DataFrame
        Fully cleaned dataframe ready for feature engineering.
    """
    logger.info("=" * 60)
    logger.info("STARTING DATA LOADING AND CLEANING PIPELINE")
    logger.info("=" * 60)

    # Step 1: Load
    df = load_raw_data(filepath)

    # Step 2: Drop unnecessary columns
    df = drop_unnecessary_columns(df)

    # Step 3: Fix TotalCharges
    df = fix_total_charges(df)

    # Step 4: Encode target
    df = encode_target(df)

    # Step 5: Validate
    validate_data(df)

    logger.info("=" * 60)
    logger.info(f"DATA LOADING COMPLETE: {df.shape[0]} rows, {df.shape[1]} columns")
    logger.info("=" * 60)

    return df
'''

# Write data_loader.py to src folder
data_loader_path = os.path.join(PROJECT_PATH, 'src', 'data_loader.py')

with open(data_loader_path, 'w') as f:
    f.write(data_loader_content)

print(f"data_loader.py written successfully to:")
print(f"  {data_loader_path}")

data_loader.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/data_loader.py


In [ ]:
# Verify if the data loader works
# Import and test data_loader
import importlib
import data_loader
importlib.reload(data_loader)

# Run the full loading pipeline
df_clean = data_loader.load_and_clean_data()

# Show the result
print("\nCleaned dataframe shape:", df_clean.shape)
print("\nData types after cleaning:")
print(df_clean.dtypes)
print("\nFirst 5 rows:")
display(df_clean.head())
print("\nChurn value counts:")
print(df_clean['Churn'].value_counts())


Cleaned dataframe shape: (7043, 20)

Data types after cleaning:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

First 5 rows:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1



Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [ ]:

data_loader_content_v2 = '''
# =============================================================================
# data_loader.py
# Responsible for loading raw data and performing initial cleaning.
# No feature engineering happens here — just getting data into a clean state.
# Version 2: Added duplicate removal step
# =============================================================================

import logging
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config

logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)


def load_raw_data(filepath: str = config.RAW_DATA_PATH) -> pd.DataFrame:
    """
    Load raw CSV data from disk.

    Parameters
    ----------
    filepath : str
        Path to the raw CSV file. Defaults to path in config.

    Returns
    -------
    pd.DataFrame
        Raw dataframe exactly as it appears in the CSV.

    Raises
    ------
    FileNotFoundError
        If the CSV file does not exist at the given path.
    """
    logger.info(f"Loading raw data from: {filepath}")

    if not os.path.exists(filepath):
        logger.error(f"Data file not found at: {filepath}")
        raise FileNotFoundError(f"No file found at {filepath}")

    df = pd.read_csv(filepath)
    logger.info(f"Raw data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")

    return df


def drop_unnecessary_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop columns that carry no predictive value.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe.

    Returns
    -------
    pd.DataFrame
        Dataframe with unnecessary columns removed.
    """
    logger.info(f"Dropping columns: {config.DROP_COLUMNS}")

    cols_to_drop = [col for col in config.DROP_COLUMNS if col in df.columns]
    df = df.drop(columns=cols_to_drop)

    logger.info(f"Remaining columns: {df.shape[1]}")
    return df


def fix_total_charges(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert TotalCharges from string to float.

    TotalCharges is stored as object dtype because some rows contain
    blank spaces instead of numbers. These correspond to new customers
    with zero tenure who have never been billed. We replace blanks
    with 0 since that is the business correct value.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with TotalCharges as object dtype.

    Returns
    -------
    pd.DataFrame
        Dataframe with TotalCharges as float64.
    """
    logger.info("Fixing TotalCharges dtype from object to float")

    blank_count = df[df["TotalCharges"].str.strip() == ""]["TotalCharges"].count()
    logger.info(f"Found {blank_count} blank TotalCharges values — replacing with 0")

    df["TotalCharges"] = df["TotalCharges"].str.strip()
    df["TotalCharges"] = df["TotalCharges"].replace("", "0")
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    remaining_nulls = df["TotalCharges"].isnull().sum()
    if remaining_nulls > 0:
        logger.warning(f"{remaining_nulls} NaN values remain in TotalCharges after conversion")
        logger.warning("Filling remaining NaN values with median")
        df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
    else:
        logger.info("TotalCharges conversion successful — no NaN values remaining")

    return df


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove duplicate rows from the dataframe.

    Duplicate rows are customers with identical values across all columns.
    In a real customer dataset this almost certainly indicates data entry
    errors rather than genuinely identical customers.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe that may contain duplicate rows.

    Returns
    -------
    pd.DataFrame
        Dataframe with duplicate rows removed.
        Only the first occurrence of each duplicate is kept.
    """
    before = df.shape[0]
    df = df.drop_duplicates(keep="first")
    after = df.shape[0]
    removed = before - after

    if removed > 0:
        logger.info(f"Removed {removed} duplicate rows: {before} -> {after} rows")
    else:
        logger.info("No duplicate rows found")

    return df


def encode_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert target column Churn from Yes/No strings to binary 1/0.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe with Churn as object dtype containing Yes/No.

    Returns
    -------
    pd.DataFrame
        Dataframe with Churn as int64 containing 1/0.
    """
    logger.info("Encoding target column: Yes -> 1, No -> 0")

    df[config.TARGET_COLUMN] = df[config.TARGET_COLUMN].map({"Yes": 1, "No": 0})

    unique_values = df[config.TARGET_COLUMN].unique()
    null_count = df[config.TARGET_COLUMN].isnull().sum()

    logger.info(f"Target column unique values after encoding: {unique_values}")
    logger.info(f"Churn rate: {df[config.TARGET_COLUMN].mean():.2%}")

    if null_count > 0:
        logger.error(f"Found {null_count} NaN values in target after encoding")
        raise ValueError(f"Target encoding failed — {null_count} unexpected values found")

    return df


def validate_data(df: pd.DataFrame) -> None:
    """
    Run basic validation checks on the cleaned dataframe.
    Logs warnings for any issues found but does not stop the pipeline.

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned dataframe to validate.

    Returns
    -------
    None
    """
    logger.info("Running data validation checks")

    # Check 1: Expected number of rows
    if df.shape[0] < 7000:
        logger.warning(f"Fewer rows than expected: {df.shape[0]} (expected ~7043)")

    # Check 2: Check for missing values
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]

    if len(cols_with_nulls) > 0:
        logger.warning("Columns with missing values after cleaning:")
        for col, count in cols_with_nulls.items():
            logger.warning(f"  {col}: {count} missing values")
    else:
        logger.info("No missing values found after cleaning")

    # Check 3: Verify target column only contains 0 and 1
    valid_target_values = set(df[config.TARGET_COLUMN].unique())
    if not valid_target_values.issubset({0, 1}):
        logger.error(f"Target column contains unexpected values: {valid_target_values}")
    else:
        logger.info("Target column validation passed")

    # Check 4: Check for remaining duplicates
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        logger.warning(f"Found {duplicate_count} duplicate rows remaining")
    else:
        logger.info("No duplicate rows found")

    # Check 5: Verify TotalCharges is now numeric
    if df["TotalCharges"].dtype != "float64":
        logger.error(f"TotalCharges dtype is {df['TotalCharges'].dtype} — expected float64")
    else:
        logger.info("TotalCharges dtype validation passed")

    logger.info("Data validation complete")


def load_and_clean_data(filepath: str = config.RAW_DATA_PATH) -> pd.DataFrame:
    """
    Master function that runs the complete data loading and cleaning pipeline.
    This is the main function called by other modules.

    Steps:
        1. Load raw CSV
        2. Drop unnecessary columns
        3. Fix TotalCharges dtype
        4. Remove duplicate rows
        5. Encode target variable
        6. Validate cleaned data

    Parameters
    ----------
    filepath : str
        Path to raw CSV file. Defaults to path in config.

    Returns
    -------
    pd.DataFrame
        Fully cleaned dataframe ready for feature engineering.
    """
    logger.info("=" * 60)
    logger.info("STARTING DATA LOADING AND CLEANING PIPELINE")
    logger.info("=" * 60)

    # Step 1: Load
    df = load_raw_data(filepath)

    # Step 2: Drop unnecessary columns
    df = drop_unnecessary_columns(df)

    # Step 3: Fix TotalCharges
    df = fix_total_charges(df)

    # Step 4: Remove duplicates
    df = remove_duplicates(df)

    # Step 5: Encode target
    df = encode_target(df)

    # Step 6: Validate
    validate_data(df)

    logger.info("=" * 60)
    logger.info(f"DATA LOADING COMPLETE: {df.shape[0]} rows, {df.shape[1]} columns")
    logger.info("=" * 60)

    return df
'''

# Overwrite the previous data_loader.py with the updated version
data_loader_path = os.path.join(PROJECT_PATH, 'src', 'data_loader.py')

with open(data_loader_path, 'w') as f:
    f.write(data_loader_content_v2)

print("data_loader.py updated successfully!")

data_loader.py updated successfully!


In [ ]:
#Verify the fix to remove duplicates
# Reload the updated module
import importlib
import data_loader
importlib.reload(data_loader)

# Run again
df_clean = data_loader.load_and_clean_data()

print("\nFinal shape:", df_clean.shape)
print("\nChurn value counts:")
print(df_clean['Churn'].value_counts())


Final shape: (7021, 20)

Churn value counts:
Churn
0    5164
1    1857
Name: count, dtype: int64


In [ ]:

# =============================================================================
# CHAPTER 4: FEATURE ENGINEERING
# Writes feature_engineering.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================

In [ ]:

feature_engineering_content = '''
# =============================================================================
# feature_engineering.py
# Custom sklearn transformer that creates all engineered features.
# Inherits from BaseEstimator and TransformerMixin so it plugs directly
# into any sklearn Pipeline without any special handling.
# =============================================================================

import logging
import numpy as np
import pandas as pd
import os
import sys

from sklearn.base import BaseEstimator, TransformerMixin

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config

logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)


class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    """
    Custom sklearn transformer that engineers all new features.

    Inheriting from BaseEstimator gives us get_params() and set_params()
    for free which are required for GridSearchCV to work correctly.

    Inheriting from TransformerMixin gives us fit_transform() for free
    which is required for sklearn Pipeline compatibility.

    This transformer is stateless — it does not learn anything from
    the training data. It just applies deterministic calculations.
    This means fit() does nothing and transform() does all the work.
    This is intentional and correct for feature engineering operations
    that are based purely on business logic and domain knowledge.

    Parameters
    ----------
    None

    Attributes
    ----------
    feature_names_out_ : list
        Names of all columns in the transformed dataframe.
        Set during fit() call.
    """

    # Define which service columns count towards TotalServices
    SERVICE_COLUMNS = [
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    # Define which values in service columns count as active
    # Anything not in this list (No, No phone service, No internet service)
    # is treated as inactive
    ACTIVE_SERVICE_VALUES = [
        "Yes",
        "DSL",
        "Fiber optic"
    ]

    # Define premium service columns
    PREMIUM_SERVICE_COLUMNS = [
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport"
    ]

    # Define automated payment methods
    AUTOMATED_PAYMENT_METHODS = [
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ]

    # Define contract risk scores — higher means higher churn risk
    CONTRACT_RISK_MAP = {
        "Month-to-month": 3,
        "One year": 2,
        "Two year": 1
    }

    # Define tenure group boundaries and labels
    TENURE_BINS = [0, 12, 24, 48, 72]
    TENURE_LABELS = ["New", "Developing", "Established", "Loyal"]


    def fit(self, X, y=None):
        """
        Fit method required by sklearn API.

        This transformer is stateless so fit() simply validates the input
        and records feature names. No parameters are learned from data.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe. Must contain all required columns.
        y : ignored
            Not used. Present for sklearn API compatibility.

        Returns
        -------
        self
            Returns self to allow method chaining (fit().transform()).
        """
        logger.info("Fitting FeatureEngineeringTransformer")

        # Validate that all required columns are present
        self._validate_columns(X)

        # Record input feature names for reference
        self.feature_names_in_ = list(X.columns)

        logger.info("FeatureEngineeringTransformer fit complete")
        return self


    def transform(self, X, y=None):
        """
        Apply all feature engineering transformations to the input dataframe.

        Creates 8 new features on top of the existing columns:
            1. TenureGroup
            2. TotalServices
            3. SpendPerService
            4. ChargesRatio
            5. HasPremiumServices
            6. IsAutomatedPayment
            7. ContractRiskScore
            8. TenureContractInteraction

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe. Must contain all required columns.
        y : ignored
            Not used. Present for sklearn API compatibility.

        Returns
        -------
        pd.DataFrame
            Transformed dataframe with 8 additional feature columns.
            Original columns are preserved.
        """
        logger.info("Starting feature engineering transformations")

        # Always work on a copy to avoid modifying the original dataframe
        X = X.copy()

        # Apply each feature engineering step
        X = self._create_tenure_group(X)
        X = self._create_total_services(X)
        X = self._create_spend_per_service(X)
        X = self._create_charges_ratio(X)
        X = self._create_has_premium_services(X)
        X = self._create_is_automated_payment(X)
        X = self._create_contract_risk_score(X)
        X = self._create_tenure_contract_interaction(X)

        # Record output feature names
        self.feature_names_out_ = list(X.columns)

        logger.info(
            f"Feature engineering complete: "
            f"{len(self.feature_names_in_)} input features -> "
            f"{len(self.feature_names_out_)} output features"
        )

        return X


    def _validate_columns(self, X):
        """
        Validate that all required columns are present in the input dataframe.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe to validate.

        Raises
        ------
        ValueError
            If any required columns are missing from the input dataframe.
        """
        required_columns = (
            self.SERVICE_COLUMNS +
            self.PREMIUM_SERVICE_COLUMNS +
            ["PaymentMethod", "Contract", "tenure",
             "MonthlyCharges", "TotalCharges"]
        )

        # Remove duplicates from required columns list
        required_columns = list(set(required_columns))

        missing = [col for col in required_columns if col not in X.columns]

        if missing:
            logger.error(f"Missing required columns: {missing}")
            raise ValueError(
                f"FeatureEngineeringTransformer requires columns: {missing}"
            )

        logger.info("Column validation passed")


    def _create_tenure_group(self, X):
        """
        Group tenure months into meaningful business categories.

        Bins:
            0-12  months -> New
            13-24 months -> Developing
            25-48 months -> Established
            49-72 months -> Loyal

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe containing tenure column.

        Returns
        -------
        pd.DataFrame
            Dataframe with TenureGroup column added.
        """
        logger.info("Creating TenureGroup")

        X["TenureGroup"] = pd.cut(
            X["tenure"],
            bins=self.TENURE_BINS,
            labels=self.TENURE_LABELS,
            include_lowest=True
        )

        # Log distribution so we can see how customers are spread
        distribution = X["TenureGroup"].value_counts().to_dict()
        logger.info(f"TenureGroup distribution: {distribution}")

        return X


    def _create_total_services(self, X):
        """
        Count how many services each customer actively uses.

        Active means the value is Yes, DSL, or Fiber optic.
        Values like No, No phone service, No internet service
        are treated as inactive and contribute 0 to the count.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe containing all service columns.

        Returns
        -------
        pd.DataFrame
            Dataframe with TotalServices column added.
        """
        logger.info("Creating TotalServices")

        # For each service column create a temporary binary column
        # 1 if the service is active, 0 otherwise
        # Then sum across all service columns for each row
        X["TotalServices"] = sum(
            X[col].isin(self.ACTIVE_SERVICE_VALUES).astype(int)
            for col in self.SERVICE_COLUMNS
        )

        logger.info(
            f"TotalServices — "
            f"min: {X['TotalServices'].min()}, "
            f"max: {X['TotalServices'].max()}, "
            f"mean: {X['TotalServices'].mean():.2f}"
        )

        return X


    def _create_spend_per_service(self, X):
        """
        Calculate monthly spend per active service.

        Formula:
            SpendPerService = MonthlyCharges / (TotalServices + 1)

        Adding 1 to TotalServices prevents division by zero for
        customers with no active services.

        Note: TotalServices must be created before calling this method.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe. Must already contain TotalServices column.

        Returns
        -------
        pd.DataFrame
            Dataframe with SpendPerService column added.
        """
        logger.info("Creating SpendPerService")

        X["SpendPerService"] = (
            X["MonthlyCharges"] / (X["TotalServices"] + 1)
        )

        logger.info(
            f"SpendPerService — "
            f"min: {X['SpendPerService'].min():.2f}, "
            f"max: {X['SpendPerService'].max():.2f}, "
            f"mean: {X['SpendPerService'].mean():.2f}"
        )

        return X


    def _create_charges_ratio(self, X):
        """
        Calculate ratio of monthly charges to total lifetime charges.

        Formula:
            ChargesRatio = MonthlyCharges / (TotalCharges + 1)

        Adding 1 to TotalCharges prevents division by zero for brand
        new customers who have never been billed (TotalCharges = 0).

        A high ratio indicates the customer is new or was recently
        upsold to a more expensive plan — both increase churn risk.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe containing MonthlyCharges and TotalCharges.

        Returns
        -------
        pd.DataFrame
            Dataframe with ChargesRatio column added.
        """
        logger.info("Creating ChargesRatio")

        X["ChargesRatio"] = (
            X["MonthlyCharges"] / (X["TotalCharges"] + 1)
        )

        logger.info(
            f"ChargesRatio — "
            f"min: {X['ChargesRatio'].min():.4f}, "
            f"max: {X['ChargesRatio'].max():.4f}, "
            f"mean: {X['ChargesRatio'].mean():.4f}"
        )

        return X


    def _create_has_premium_services(self, X):
        """
        Create binary flag for whether customer has any premium add-ons.

        Premium services are OnlineSecurity, OnlineBackup,
        DeviceProtection, and TechSupport.

        A value of Yes in any of these columns means the customer
        has invested in premium services and is less likely to churn.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe containing premium service columns.

        Returns
        -------
        pd.DataFrame
            Dataframe with HasPremiumServices column added.
            1 = has at least one premium service
            0 = has no premium services
        """
        logger.info("Creating HasPremiumServices")

        X["HasPremiumServices"] = (
            X[self.PREMIUM_SERVICE_COLUMNS]
            .eq("Yes")
            .any(axis=1)
            .astype(int)
        )

        distribution = X["HasPremiumServices"].value_counts().to_dict()
        logger.info(f"HasPremiumServices distribution: {distribution}")

        return X


    def _create_is_automated_payment(self, X):
        """
        Create binary flag for whether customer pays automatically.

        Automated payment methods are:
            Bank transfer (automatic)
            Credit card (automatic)

        Manual payment methods are:
            Electronic check
            Mailed check

        Customers on automated payment are passively committed
        and less likely to churn.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe containing PaymentMethod column.

        Returns
        -------
        pd.DataFrame
            Dataframe with IsAutomatedPayment column added.
            1 = automated payment
            0 = manual payment
        """
        logger.info("Creating IsAutomatedPayment")

        X["IsAutomatedPayment"] = (
            X["PaymentMethod"]
            .isin(self.AUTOMATED_PAYMENT_METHODS)
            .astype(int)
        )

        distribution = X["IsAutomatedPayment"].value_counts().to_dict()
        logger.info(f"IsAutomatedPayment distribution: {distribution}")

        return X


    def _create_contract_risk_score(self, X):
        """
        Encode contract type as an ordinal churn risk score.

        Risk scores:
            Month-to-month -> 3  (highest churn risk)
            One year       -> 2  (medium churn risk)
            Two year       -> 1  (lowest churn risk)

        This encoding uses domain knowledge to impose a meaningful
        ordering on contract types rather than treating them as
        unordered categories.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe containing Contract column.

        Returns
        -------
        pd.DataFrame
            Dataframe with ContractRiskScore column added.

        Raises
        ------
        ValueError
            If any unexpected contract values are found.
        """
        logger.info("Creating ContractRiskScore")

        # Check for unexpected contract values before mapping
        unexpected = set(X["Contract"].unique()) - set(self.CONTRACT_RISK_MAP.keys())
        if unexpected:
            logger.error(f"Unexpected contract values found: {unexpected}")
            raise ValueError(f"Unknown contract types: {unexpected}")

        X["ContractRiskScore"] = X["Contract"].map(self.CONTRACT_RISK_MAP)

        distribution = X["ContractRiskScore"].value_counts().to_dict()
        logger.info(f"ContractRiskScore distribution: {distribution}")

        return X


    def _create_tenure_contract_interaction(self, X):
        """
        Create interaction feature combining tenure and contract risk.

        Formula:
            TenureContractInteraction = tenure * ContractRiskScore

        This captures the combined effect of loyalty and commitment.

        High tenure + low risk score (Two year) = very safe customer
        Low tenure  + high risk score (Month-to-month) = very risky customer

        Note: ContractRiskScore must be created before calling this method.

        Parameters
        ----------
        X : pd.DataFrame
            Input dataframe. Must already contain ContractRiskScore.

        Returns
        -------
        pd.DataFrame
            Dataframe with TenureContractInteraction column added.
        """
        logger.info("Creating TenureContractInteraction")

        X["TenureContractInteraction"] = (
            X["tenure"] * X["ContractRiskScore"]
        )

        logger.info(
            f"TenureContractInteraction — "
            f"min: {X['TenureContractInteraction'].min()}, "
            f"max: {X['TenureContractInteraction'].max()}, "
            f"mean: {X['TenureContractInteraction'].mean():.2f}"
        )

        return X


    def get_feature_names_out(self):
        """
        Return names of all output features after transformation.

        Required by sklearn API for pipeline compatibility.

        Returns
        -------
        list
            List of all column names in the transformed dataframe.

        Raises
        ------
        AttributeError
            If transform() has not been called yet.
        """
        if not hasattr(self, "feature_names_out_"):
            raise AttributeError(
                "feature_names_out_ is not set. "
                "Call fit_transform() or transform() first."
            )
        return self.feature_names_out_
'''

# Write feature_engineering.py to src folder
feature_engineering_path = os.path.join(PROJECT_PATH, 'src', 'feature_engineering.py')

with open(feature_engineering_path, 'w') as f:
    f.write(feature_engineering_content)

print(f"feature_engineering.py written successfully to:")
print(f"  {feature_engineering_path}")

feature_engineering.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/feature_engineering.py


In [ ]:
#Verifying if feature engineering works
import importlib
import feature_engineering
importlib.reload(feature_engineering)

from feature_engineering import FeatureEngineeringTransformer

# Use the clean dataframe from data_loader
transformer = FeatureEngineeringTransformer()
df_engineered = transformer.fit_transform(df_clean)

# Show results
print("Shape before engineering:", df_clean.shape)
print("Shape after engineering: ", df_engineered.shape)
print(f"\nNew features added: {df_engineered.shape[1] - df_clean.shape[1]}")

print("\nNew columns created:")
new_cols = [col for col in df_engineered.columns if col not in df_clean.columns]
for col in new_cols:
    print(f"  {col}")

print("\nSample of engineered features for first 5 customers:")
display(df_engineered[new_cols].head())

print("\nNo missing values in engineered features:")
print(df_engineered[new_cols].isnull().sum())

Shape before engineering: (7021, 20)
Shape after engineering:  (7021, 28)

New features added: 8

New columns created:
  TenureGroup
  TotalServices
  SpendPerService
  ChargesRatio
  HasPremiumServices
  IsAutomatedPayment
  ContractRiskScore
  TenureContractInteraction

Sample of engineered features for first 5 customers:


,TenureGroup,TotalServices,SpendPerService,ChargesRatio,HasPremiumServices,IsAutomatedPayment,ContractRiskScore,TenureContractInteraction
0,New,2,9.950000,0.967585,1,0,3,3
1,Established,4,11.390000,0.030124,1,0,2,68
2,New,4,10.770000,0.493358,1,0,3,6
3,Established,4,8.460000,0.022967,1,1,2,90
4,New,2,23.566667,0.463151,0,0,3,6



No missing values in engineered features:
TenureGroup                  0
TotalServices                0
SpendPerService              0
ChargesRatio                 0
HasPremiumServices           0
IsAutomatedPayment           0
ContractRiskScore            0
TenureContractInteraction    0
dtype: int64


In [ ]:

# =============================================================================
# CHAPTER 5: PREPROCESSING
# Writes preprocessing.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================

In [ ]:

preprocessing_content = '''
# =============================================================================
# preprocessing.py
# Builds the full sklearn preprocessing pipeline.
# Handles imputation, encoding, scaling, and train/test splitting.
# The pipeline is designed to prevent data leakage at every step.
# =============================================================================

import logging
import numpy as np
import pandas as pd
import os
import sys

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config
from feature_engineering import FeatureEngineeringTransformer

logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)


# =============================================================================
# COLUMN DEFINITIONS
# These define exactly which columns go into which pipeline
# Must match the output of FeatureEngineeringTransformer exactly
# =============================================================================

# Numerical columns — will be imputed with median then scaled
NUMERICAL_COLUMNS = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "TotalServices",
    "SpendPerService",
    "ChargesRatio",
    "ContractRiskScore",
    "TenureContractInteraction"
]

# Binary categorical columns — will be imputed then ordinally encoded
BINARY_COLUMNS = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "HasPremiumServices",
    "IsAutomatedPayment"
]

# SeniorCitizen is int but categorical — treat as binary
# Already 0/1 so OrdinalEncoder passes it through unchanged
SENIOR_CITIZEN_COLUMN = ["SeniorCitizen"]

# Multi-class categorical columns — will be one-hot encoded
MULTICLASS_COLUMNS = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
    "TenureGroup"
]


def split_features_target(df: pd.DataFrame):
    """
    Separate the dataframe into features (X) and target (y).

    Parameters
    ----------
    df : pd.DataFrame
        Full cleaned and engineered dataframe including target column.

    Returns
    -------
    X : pd.DataFrame
        Feature matrix — all columns except target.
    y : pd.Series
        Target vector — Churn column only.
    """
    logger.info(f"Splitting features and target — target column: {config.TARGET_COLUMN}")

    X = df.drop(columns=[config.TARGET_COLUMN])
    y = df[config.TARGET_COLUMN]

    logger.info(f"Features shape: {X.shape}")
    logger.info(f"Target shape: {y.shape}")
    logger.info(f"Target distribution — 0: {(y==0).sum()}, 1: {(y==1).sum()}")

    return X, y


def split_train_test(X: pd.DataFrame, y: pd.Series):
    """
    Split data into training and test sets.

    Uses stratified splitting to ensure both sets have the same
    class distribution as the full dataset. This is critical for
    imbalanced datasets like churn where random splitting could
    produce an unrepresentative test set.

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix.
    y : pd.Series
        Target vector.

    Returns
    -------
    X_train : pd.DataFrame
    X_test  : pd.DataFrame
    y_train : pd.Series
    y_test  : pd.Series
    """
    logger.info(
        f"Splitting data — "
        f"test size: {config.TEST_SIZE}, "
        f"random seed: {config.RANDOM_SEED}, "
        f"stratified: True"
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_SEED,
        stratify=y        # Ensures same churn rate in both splits
    )

    logger.info(f"Training set: {X_train.shape[0]} rows")
    logger.info(f"Test set    : {X_test.shape[0]} rows")
    logger.info(
        f"Training churn rate: {y_train.mean():.2%} | "
        f"Test churn rate: {y_test.mean():.2%}"
    )

    return X_train, X_test, y_train, y_test


def build_numerical_pipeline() -> Pipeline:
    """
    Build pipeline for numerical features.

    Steps:
        1. SimpleImputer  — fills missing values with column median
                            Median is robust to outliers unlike mean
        2. StandardScaler — scales to mean=0, std=1
                            Required for logistic regression to converge properly

    Returns
    -------
    Pipeline
        Fitted-ready sklearn pipeline for numerical columns.
    """
    logger.info("Building numerical pipeline")

    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])


def build_binary_pipeline() -> Pipeline:
    """
    Build pipeline for binary categorical features.

    Steps:
        1. SimpleImputer   — fills missing values with most frequent value
        2. OrdinalEncoder  — encodes two-value categories as 0 and 1
                             Safe for binary columns — no dummy variable trap

    Returns
    -------
    Pipeline
        Fitted-ready sklearn pipeline for binary categorical columns.
    """
    logger.info("Building binary categorical pipeline")

    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1    # Unknown categories get -1 in production
        ))
    ])


def build_multiclass_pipeline() -> Pipeline:
    """
    Build pipeline for multi-class categorical features.

    Steps:
        1. SimpleImputer  — fills missing values with most frequent value
        2. OneHotEncoder  — creates binary column per category
                            drop=first prevents dummy variable trap
                            handle_unknown=ignore is safe for production

    Returns
    -------
    Pipeline
        Fitted-ready sklearn pipeline for multi-class categorical columns.
    """
    logger.info("Building multi-class categorical pipeline")

    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            drop="first",               # Prevents dummy variable trap
            handle_unknown="ignore",    # Unknown categories become all zeros
            sparse_output=False         # Return dense array not sparse matrix
        ))
    ])


def build_preprocessor() -> ColumnTransformer:
    """
    Build the full preprocessing ColumnTransformer.

    Combines all three sub-pipelines and applies each to its
    designated set of columns. ColumnTransformer handles the
    column routing and concatenates the results automatically.

    Returns
    -------
    ColumnTransformer
        Full preprocessing transformer ready to be fitted.
    """
    logger.info("Building full ColumnTransformer preprocessor")

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                build_numerical_pipeline(),
                NUMERICAL_COLUMNS
            ),
            (
                "binary",
                build_binary_pipeline(),
                BINARY_COLUMNS + SENIOR_CITIZEN_COLUMN
            ),
            (
                "multiclass",
                build_multiclass_pipeline(),
                MULTICLASS_COLUMNS
            )
        ],
        remainder="drop",       # Drop any columns not explicitly listed
        verbose_feature_names_out=False   # Cleaner feature names
    )

    return preprocessor


def build_full_pipeline() -> Pipeline:
    """
    Build the complete end-to-end pipeline including feature engineering
    and preprocessing.

    This pipeline takes raw cleaned data as input and outputs a fully
    preprocessed numerical matrix ready for model training.

    Note: SMOTE is NOT included here because it must only be applied
    to training data inside the cross validation loop. Adding SMOTE
    here would apply it to validation folds which would cause data
    leakage.

    Pipeline steps:
        1. FeatureEngineeringTransformer — creates 8 new features
        2. ColumnTransformer             — encodes and scales all features

    Returns
    -------
    Pipeline
        Complete preprocessing pipeline ready to be fitted on training data.
    """
    logger.info("Building full end-to-end preprocessing pipeline")

    pipeline = Pipeline(steps=[
        ("feature_engineering", FeatureEngineeringTransformer()),
        ("preprocessor", build_preprocessor())
    ])

    logger.info("Full pipeline built successfully")
    return pipeline


def fit_transform_pipeline(pipeline: Pipeline,
                           X_train: pd.DataFrame,
                           X_test: pd.DataFrame):
    """
    Fit the pipeline on training data only then transform both sets.

    This is the critical step that prevents data leakage.
    The pipeline learns its parameters (scaler means, encoder categories)
    exclusively from training data. The test set is only transformed
    using those learned parameters — never used to fit anything.

    Parameters
    ----------
    pipeline : Pipeline
        Unfitted preprocessing pipeline from build_full_pipeline().
    X_train : pd.DataFrame
        Training features. Pipeline is fitted on this.
    X_test : pd.DataFrame
        Test features. Only transformed, never fitted.

    Returns
    -------
    X_train_processed : np.ndarray
        Processed training features ready for SMOTE and model training.
    X_test_processed : np.ndarray
        Processed test features ready for final evaluation.
    pipeline : Pipeline
        Fitted pipeline — saved to disk for production use.
    """
    logger.info("Fitting pipeline on training data only")
    X_train_processed = pipeline.fit_transform(X_train)
    logger.info(f"Training data processed: {X_train_processed.shape}")

    logger.info("Transforming test data using fitted pipeline")
    X_test_processed = pipeline.transform(X_test)
    logger.info(f"Test data processed: {X_test_processed.shape}")

    return X_train_processed, X_test_processed, pipeline


def run_preprocessing_pipeline(df: pd.DataFrame):
    """
    Master function that runs the complete preprocessing pipeline.

    This is the main function called by train.py.

    Steps:
        1. Separate features and target
        2. Split into train and test sets
        3. Build the full pipeline
        4. Fit on training data, transform both sets

    Parameters
    ----------
    df : pd.DataFrame
        Fully cleaned dataframe from data_loader.load_and_clean_data().

    Returns
    -------
    X_train_processed : np.ndarray
        Processed training features.
    X_test_processed  : np.ndarray
        Processed test features.
    y_train           : pd.Series
        Training labels.
    y_test            : pd.Series
        Test labels.
    pipeline          : Pipeline
        Fitted preprocessing pipeline.
    """
    logger.info("=" * 60)
    logger.info("STARTING PREPROCESSING PIPELINE")
    logger.info("=" * 60)

    # Step 1: Separate features and target
    X, y = split_features_target(df)

    # Step 2: Split into train and test
    X_train, X_test, y_train, y_test = split_train_test(X, y)

    # Step 3: Build the pipeline
    pipeline = build_full_pipeline()

    # Step 4: Fit on train, transform both
    X_train_processed, X_test_processed, pipeline = fit_transform_pipeline(
        pipeline, X_train, X_test
    )

    logger.info("=" * 60)
    logger.info("PREPROCESSING COMPLETE")
    logger.info(f"X_train_processed shape: {X_train_processed.shape}")
    logger.info(f"X_test_processed shape : {X_test_processed.shape}")
    logger.info(f"y_train shape          : {y_train.shape}")
    logger.info(f"y_test shape           : {y_test.shape}")
    logger.info("=" * 60)

    return X_train_processed, X_test_processed, y_train, y_test, pipeline
'''

# Write preprocessing.py to src folder
preprocessing_path = os.path.join(PROJECT_PATH, 'src', 'preprocessing.py')

with open(preprocessing_path, 'w') as f:
    f.write(preprocessing_content)

print(f"preprocessing.py written successfully to:")
print(f"  {preprocessing_path}")

preprocessing.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/preprocessing.py


In [ ]:
#Verify that preprocessing is working
import importlib
import preprocessing
importlib.reload(preprocessing)

# Run the full preprocessing pipeline
X_train_processed, X_test_processed, y_train, y_test, pipeline = \
    preprocessing.run_preprocessing_pipeline(df_clean)

# Show results
print("\nPreprocessing Results:")
print(f"  X_train_processed : {X_train_processed.shape}")
print(f"  X_test_processed  : {X_test_processed.shape}")
print(f"  y_train           : {y_train.shape}")
print(f"  y_test            : {y_test.shape}")

print(f"\nTraining churn rate : {y_train.mean():.2%}")
print(f"Test churn rate     : {y_test.mean():.2%}")

print("\nFirst 3 rows of processed training data:")
print(X_train_processed[:3])

print("\nPipeline steps:")
for step_name, step in pipeline.steps:
    print(f"  {step_name}: {type(step).__name__}")


Preprocessing Results:
  X_train_processed : (5616, 40)
  X_test_processed  : (1405, 40)
  y_train           : (5616,)
  y_test            : (1405,)

Training churn rate : 26.44%
Test churn rate     : 26.48%

First 3 rows of processed training data:
[[-1.24133083  0.19316451 -0.94471422 -0.93192967  3.04567307  0.15930037
   0.83141266 -1.21011777  1.          0.          0.          1.
   0.          0.          1.          0.          0.          0.
   1.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.          0.
   0.          0.          0.          1.        ]
 [-0.71107812  0.64735514 -0.43406164  0.3640795   0.39464353 -0.08404918
   0.83141266 -0.35544873  0.          0.          0.          1.
   1.          1.          1.          0.          0.          1.
   1.          0.          0.          1.          0.          0.
   0.          1.     

In [ ]:

# =============================================================================
# CHAPTER 6: TRAINING
# Writes train.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================

In [ ]:

train_content = '''
# =============================================================================
# train.py
# Handles SMOTE balancing, hyperparameter tuning via GridSearchCV,
# final model training, and saving the fitted pipeline to disk.
# =============================================================================

import logging
import numpy as np
import pandas as pd
import os
import sys
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config

logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)


def apply_smote(X_train: np.ndarray, y_train: pd.Series):
    """
    Apply SMOTE to balance the training data.

    SMOTE creates synthetic minority class samples by interpolating
    between existing minority class samples and their nearest neighbours.
    This gives the model more diverse churner examples to learn from.

    IMPORTANT: Only ever apply SMOTE to training data.
    Never apply to validation or test data.

    Parameters
    ----------
    X_train : np.ndarray
        Processed training features from preprocessing pipeline.
    y_train : pd.Series
        Training labels — imbalanced (roughly 74% / 26% split).

    Returns
    -------
    X_train_balanced : np.ndarray
        Balanced training features with synthetic samples added.
    y_train_balanced : np.ndarray
        Balanced training labels — now 50% / 50% split.
    """
    logger.info("Applying SMOTE to balance training data")
    logger.info(
        f"Before SMOTE — "
        f"Class 0: {(y_train==0).sum()}, "
        f"Class 1: {(y_train==1).sum()}"
    )

    smote = SMOTE(
        sampling_strategy=config.SMOTE_SAMPLING_STRATEGY,
        k_neighbors=config.SMOTE_K_NEIGHBOURS,
        random_state=config.RANDOM_SEED
    )

    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

    logger.info(
        f"After SMOTE — "
        f"Class 0: {(y_train_balanced==0).sum()}, "
        f"Class 1: {(y_train_balanced==1).sum()}"
    )
    logger.info(f"Training set size after SMOTE: {X_train_balanced.shape[0]} rows")

    return X_train_balanced, y_train_balanced


def build_model_pipeline() -> ImbPipeline:
    """
    Build the model pipeline that includes SMOTE and LogisticRegression.

    We use imblearn Pipeline instead of sklearn Pipeline because
    imblearn Pipeline correctly handles resamplers like SMOTE.
    It applies SMOTE only during fit() and skips it during transform()
    and predict() — exactly what we need for safe cross validation.

    Pipeline steps:
        1. SMOTE              — balances classes on training folds only
        2. LogisticRegression — the classifier

    Returns
    -------
    ImbPipeline
        Unfitted model pipeline ready for GridSearchCV.
    """
    logger.info("Building model pipeline with SMOTE and LogisticRegression")

    model_pipeline = ImbPipeline(steps=[
        (
            "smote",
            SMOTE(
                sampling_strategy=config.SMOTE_SAMPLING_STRATEGY,
                k_neighbors=config.SMOTE_K_NEIGHBOURS,
                random_state=config.RANDOM_SEED
            )
        ),
        (
            "logisticregression",
            LogisticRegression(
                class_weight=config.CLASS_WEIGHT,
                max_iter=config.MAX_ITER,
                random_state=config.RANDOM_SEED
            )
        )
    ])

    return model_pipeline


def build_cross_validator() -> StratifiedKFold:
    """
    Build the cross validator.

    StratifiedKFold maintains the same class distribution in each fold
    as the full training set. This is essential for imbalanced datasets
    because random splits might produce folds with very few churners.

    Returns
    -------
    StratifiedKFold
        Cross validator with settings from config.
    """
    logger.info(
        f"Building StratifiedKFold cross validator — "
        f"{config.CV_FOLDS} folds"
    )

    return StratifiedKFold(
        n_splits=config.CV_FOLDS,
        shuffle=True,
        random_state=config.RANDOM_SEED
    )


def run_hyperparameter_tuning(
    X_train_balanced: np.ndarray,
    y_train_balanced: np.ndarray
) -> GridSearchCV:
    """
    Run GridSearchCV to find the best hyperparameters.

    For each combination in HYPERPARAMETER_GRID runs 5-fold
    cross validation and records the average ROC AUC score.
    Picks the combination with the highest average score.

    Note: We pass balanced data from SMOTE here because GridSearchCV
    will run cross validation on whatever data we pass it. Since we
    are using imblearn Pipeline with SMOTE inside it, SMOTE will be
    applied correctly on each training fold during cross validation.
    This means we pass the original unbalanced training data here
    and let the pipeline handle SMOTE internally.

    Parameters
    ----------
    X_train_balanced : np.ndarray
        Processed training features — NOT yet SMOTE balanced.
        SMOTE is handled inside the pipeline during cross validation.
    y_train_balanced : np.ndarray
        Training labels — NOT yet SMOTE balanced.

    Returns
    -------
    GridSearchCV
        Fitted GridSearchCV object containing best estimator,
        best parameters, and full cross validation results.
    """
    logger.info("=" * 60)
    logger.info("STARTING HYPERPARAMETER TUNING")
    logger.info(f"Parameter grid: {config.HYPERPARAMETER_GRID}")
    logger.info(
        f"Total combinations: "
        f"{len(config.HYPERPARAMETER_GRID['logisticregression__C']) * len(config.HYPERPARAMETER_GRID['logisticregression__penalty']) * len(config.HYPERPARAMETER_GRID['logisticregression__solver'])}"
    )
    logger.info(f"Scoring metric: {config.SCORING_METRIC}")
    logger.info("=" * 60)

    model_pipeline = build_model_pipeline()
    cv = build_cross_validator()

    grid_search = GridSearchCV(
        estimator=model_pipeline,
        param_grid=config.HYPERPARAMETER_GRID,
        cv=cv,
        scoring=config.SCORING_METRIC,
        n_jobs=-1,          # Use all available CPU cores
        verbose=2,          # Show progress during tuning
        refit=True,         # Refit best model on full training data
        return_train_score=True  # Track training scores for overfitting check
    )

    grid_search.fit(X_train_balanced, y_train_balanced)

    logger.info("=" * 60)
    logger.info("HYPERPARAMETER TUNING COMPLETE")
    logger.info(f"Best parameters : {grid_search.best_params_}")
    logger.info(f"Best ROC AUC    : {grid_search.best_score_:.4f}")
    logger.info("=" * 60)

    return grid_search


def log_cv_results(grid_search: GridSearchCV) -> None:
    """
    Log the top cross validation results for transparency.

    Shows the top 5 parameter combinations ranked by mean test score.
    Also checks for overfitting by comparing train and test scores.

    Parameters
    ----------
    grid_search : GridSearchCV
        Fitted GridSearchCV object.

    Returns
    -------
    None
    """
    results = pd.DataFrame(grid_search.cv_results_)
    results = results.sort_values("mean_test_score", ascending=False)

    logger.info("Top 5 hyperparameter combinations:")
    top5 = results[[
        "param_logisticregression__C",
        "param_logisticregression__penalty",
        "param_logisticregression__solver",
        "mean_train_score",
        "mean_test_score",
        "std_test_score"
    ]].head()

    for _, row in top5.iterrows():
        logger.info(
            f"  C={row['param_logisticregression__C']:<8} "
            f"penalty={row['param_logisticregression__penalty']:<3} "
            f"solver={row['param_logisticregression__solver']:<12} "
            f"train={row['mean_train_score']:.4f} "
            f"val={row['mean_test_score']:.4f} "
            f"std={row['std_test_score']:.4f}"
        )

    # Overfitting check
    best_train = results.iloc[0]["mean_train_score"]
    best_val = results.iloc[0]["mean_test_score"]
    gap = best_train - best_val

    logger.info(f"Overfitting check — train: {best_train:.4f}, val: {best_val:.4f}, gap: {gap:.4f}")

    if gap > 0.05:
        logger.warning(f"Potential overfitting detected — train/val gap is {gap:.4f}")
    else:
        logger.info("No significant overfitting detected")


def save_pipeline(pipeline, path: str = config.MODEL_PATH) -> None:
    """
    Save the fitted pipeline to disk using joblib.

    joblib is preferred over pickle for sklearn objects because it
    handles numpy arrays more efficiently.

    The saved pipeline includes:
        - FeatureEngineeringTransformer
        - ColumnTransformer (with fitted scaler and encoders)
        - SMOTE
        - LogisticRegression (with fitted coefficients)

    Loading this single file in production gives you a complete
    end-to-end pipeline that takes raw customer data and returns
    a churn probability.

    Parameters
    ----------
    pipeline : fitted sklearn/imblearn Pipeline
        The complete fitted pipeline to save.
    path : str
        File path to save the pipeline. Defaults to config.MODEL_PATH.

    Returns
    -------
    None
    """
    # Create models directory if it does not exist
    os.makedirs(os.path.dirname(path), exist_ok=True)

    joblib.dump(pipeline, path)
    logger.info(f"Pipeline saved to: {path}")

    # Log file size so we know how large the model is
    file_size_mb = os.path.getsize(path) / (1024 * 1024)
    logger.info(f"Saved pipeline size: {file_size_mb:.2f} MB")


def run_training_pipeline(
    X_train_processed: np.ndarray,
    X_test_processed: np.ndarray,
    y_train: pd.Series,
    y_test: pd.Series,
    preprocessing_pipeline
):
    """
    Master function that runs the complete training pipeline.

    This is the main function called by main.py.

    Steps:
        1. Run hyperparameter tuning with cross validation
        2. Log cross validation results
        3. Extract best model
        4. Save the best model pipeline to disk

    Note: We do NOT apply SMOTE before passing to GridSearchCV.
    SMOTE is inside the imblearn Pipeline and will be applied
    correctly on each training fold during cross validation.

    Parameters
    ----------
    X_train_processed : np.ndarray
        Processed training features from preprocessing pipeline.
    X_test_processed : np.ndarray
        Processed test features from preprocessing pipeline.
    y_train : pd.Series
        Training labels.
    y_test : pd.Series
        Test labels.
    preprocessing_pipeline : Pipeline
        Fitted preprocessing pipeline — saved alongside model.

    Returns
    -------
    best_model : fitted ImbPipeline
        Best model pipeline from GridSearchCV.
    grid_search : GridSearchCV
        Full GridSearchCV results for analysis.
    """
    logger.info("=" * 60)
    logger.info("STARTING TRAINING PIPELINE")
    logger.info("=" * 60)

    # Step 1: Run hyperparameter tuning
    # Pass unbalanced training data — SMOTE handled inside pipeline
    grid_search = run_hyperparameter_tuning(
        X_train_processed, y_train
    )

    # Step 2: Log results
    log_cv_results(grid_search)

    # Step 3: Extract best model
    best_model = grid_search.best_estimator_
    logger.info(f"Best model extracted: {best_model}")

    # Step 4: Save the best model
    save_pipeline(best_model, config.MODEL_PATH)

    logger.info("=" * 60)
    logger.info("TRAINING PIPELINE COMPLETE")
    logger.info("=" * 60)

    return best_model, grid_search
'''

# Write train.py to src folder
train_path = os.path.join(PROJECT_PATH, 'src', 'train.py')

with open(train_path, 'w') as f:
    f.write(train_content)

print(f"train.py written successfully to:")
print(f"  {train_path}")

train.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/train.py


In [ ]:
# Making sure imbalanced_learn is installed
!pip install imbalanced-learn --quiet
print("imbalanced-learn installed successfully!")

imbalanced-learn installed successfully!


In [ ]:
#Verifying if training works
import importlib
import train
importlib.reload(train)

# Run the full training pipeline
# Warning: This will take a few minutes because GridSearchCV
# is running 120 model fits
print("Starting training — this will take a few minutes...")
print("You will see progress output from GridSearchCV below")
print("=" * 60)

best_model, grid_search = train.run_training_pipeline(
    X_train_processed,
    X_test_processed,
    y_train,
    y_test,
    pipeline
)

print("=" * 60)
print("Training complete!")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best ROC AUC    : {grid_search.best_score_:.4f}")

In [31]:

# Update MAX_ITER from 1000 to 3000
config_path = os.path.join(PROJECT_PATH, 'src', 'config.py')

with open(config_path, 'r') as f:
    config_content = f.read()

config_content = config_content.replace(
    'MAX_ITER = 1000',
    'MAX_ITER = 3000'
)

with open(config_path, 'w') as f:
    f.write(config_content)

print("config.py updated — MAX_ITER changed from 1000 to 3000")

config.py updated — MAX_ITER changed from 1000 to 3000


In [32]:
#Delete the old saved model
# Force retraining with new MAX_ITER
import config
importlib.reload(config)

if os.path.exists(config.MODEL_PATH):
    os.remove(config.MODEL_PATH)
    print("Old model deleted — will retrain with MAX_ITER=3000")
else:
    print("No saved model found")

Old model deleted — will retrain with MAX_ITER=3000


In [37]:
# Verify if model waa saved
# Check model file exists and get details
if os.path.exists(config.MODEL_PATH):
    file_size_mb = os.path.getsize(config.MODEL_PATH) / (1024 * 1024)
    print("Model saved successfully!")
    print(f"Location : {config.MODEL_PATH}")
    print(f"Size     : {file_size_mb:.2f} MB")
else:
    print("Model file not found!")

Model saved successfully!
Location : /content/drive/MyDrive/telecom_churn_prediction/models/logistic_regression_pipeline.pkl
Size     : 0.45 MB


In [38]:
# Verify if next session will load the saved model
# Simulate what happens next session
import joblib

loaded_model = joblib.load(config.MODEL_PATH)
print("Model loads correctly!")
print(f"Model type : {type(loaded_model).__name__}")
print(f"Model steps: {[step[0] for step in loaded_model.steps]}")

Model loads correctly!
Model type : Pipeline
Model steps: ['smote', 'logisticregression']


In [40]:

# =============================================================================
# CHAPTER 7: EVALUATION
# Writes evaluate.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================

In [41]:

evaluate_content = '''
# =============================================================================
# evaluate.py
# Produces all evaluation metrics and plots for the trained model.
# Covers confusion matrix, classification report, ROC AUC curve,
# precision recall curve, and full SHAP interpretability analysis.
# =============================================================================

import logging
import numpy as np
import pandas as pd
import os
import sys
import joblib
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

import shap

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config

logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)

# Set consistent plot style across all figures
sns.set_theme(style="whitegrid", palette="husl")


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def save_figure(fig, filename: str) -> None:
    """
    Save a matplotlib figure to the reports/figures directory.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure to save.
    filename : str
        Filename including extension e.g. confusion_matrix.png

    Returns
    -------
    None
    """
    os.makedirs(config.FIGURES_DIR, exist_ok=True)
    path = os.path.join(config.FIGURES_DIR, filename)
    fig.savefig(path, dpi=config.FIGURE_DPI, bbox_inches="tight")
    logger.info(f"Figure saved: {path}")
    plt.close(fig)


def get_predictions(model, X_test: np.ndarray):
    """
    Generate both class predictions and probability predictions.

    Parameters
    ----------
    model : fitted Pipeline
        Trained model pipeline.
    X_test : np.ndarray
        Processed test features.

    Returns
    -------
    y_pred : np.ndarray
        Binary class predictions using default threshold 0.5
    y_prob : np.ndarray
        Probability of churn for each customer.
        Used for ROC AUC and threshold analysis.
    """
    logger.info("Generating predictions on test set")

    # predict() uses threshold 0.5 by default
    y_pred = model.predict(X_test)

    # predict_proba() returns probabilities for both classes
    # [:, 1] selects the probability of class 1 (churn)
    y_prob = model.predict_proba(X_test)[:, 1]

    logger.info(f"Predictions generated for {len(y_pred)} customers")
    logger.info(f"Predicted churn rate: {y_pred.mean():.2%}")

    return y_pred, y_prob


# =============================================================================
# CONFUSION MATRIX
# =============================================================================

def plot_confusion_matrix(y_test, y_pred) -> dict:
    """
    Plot and save the confusion matrix.

    Also logs the business interpretation of each cell:
        TN: correctly identified loyal customers
        FP: loyal customers incorrectly flagged for retention offers
        FN: churners who slipped through undetected (most costly mistake)
        TP: churners correctly identified for intervention

    Parameters
    ----------
    y_test : pd.Series
        True labels from test set.
    y_pred : np.ndarray
        Predicted labels from model.

    Returns
    -------
    dict
        Dictionary containing TN, FP, FN, TP values.
    """
    logger.info("Plotting confusion matrix")

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    logger.info("Confusion matrix results:")
    logger.info(f"  True Negatives  (correctly identified loyal)   : {tn}")
    logger.info(f"  False Positives (loyal flagged as churner)      : {fp}")
    logger.info(f"  False Negatives (churners missed by model)      : {fn}")
    logger.info(f"  True Positives  (churners correctly identified) : {tp}")

    # Plot
    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Predicted Stay", "Predicted Churn"],
        yticklabels=["Actually Stay", "Actually Churn"],
        ax=ax,
        linewidths=0.5
    )

    ax.set_title("Confusion Matrix — Telecom Churn Prediction", fontsize=14, pad=15)
    ax.set_ylabel("Actual Label", fontsize=12)
    ax.set_xlabel("Predicted Label", fontsize=12)

    # Add business interpretation as text below the plot
    interpretation = (
        f"True Negatives: {tn} | False Positives: {fp} | "
        f"False Negatives: {fn} | True Positives: {tp}"
    )
    fig.text(
        0.5, -0.02, interpretation,
        ha="center", fontsize=10,
        style="italic", color="gray"
    )

    save_figure(fig, "confusion_matrix.png")

    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}


# =============================================================================
# CLASSIFICATION REPORT
# =============================================================================

def log_classification_report(y_test, y_pred) -> dict:
    """
    Generate and log the full classification report.

    Parameters
    ----------
    y_test : pd.Series
        True labels.
    y_pred : np.ndarray
        Predicted labels.

    Returns
    -------
    dict
        Classification report as dictionary for saving to results file.
    """
    logger.info("Classification report:")

    report = classification_report(
        y_test, y_pred,
        target_names=["Stayed (0)", "Churned (1)"],
        output_dict=False
    )

    report_dict = classification_report(
        y_test, y_pred,
        target_names=["Stayed (0)", "Churned (1)"],
        output_dict=True
    )

    # Print full report
    print("\\n" + "=" * 60)
    print("CLASSIFICATION REPORT")
    print("=" * 60)
    print(report)
    print("=" * 60)

    # Log key metrics for churners specifically
    churn_precision = report_dict["Churned (1)"]["precision"]
    churn_recall    = report_dict["Churned (1)"]["recall"]
    churn_f1        = report_dict["Churned (1)"]["f1-score"]

    logger.info(f"Churn class metrics:")
    logger.info(f"  Precision : {churn_precision:.4f}")
    logger.info(f"  Recall    : {churn_recall:.4f}")
    logger.info(f"  F1 Score  : {churn_f1:.4f}")

    return report_dict


# =============================================================================
# ROC AUC CURVE
# =============================================================================

def plot_roc_curve(y_test, y_prob) -> float:
    """
    Plot and save the ROC AUC curve.

    The ROC curve plots true positive rate (recall) against false
    positive rate across all possible classification thresholds.
    The AUC summarises this as a single number between 0.5 and 1.0.

    Parameters
    ----------
    y_test : pd.Series
        True labels.
    y_prob : np.ndarray
        Predicted churn probabilities.

    Returns
    -------
    float
        ROC AUC score.
    """
    logger.info("Plotting ROC AUC curve")

    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)

    logger.info(f"ROC AUC Score: {auc_score:.4f}")

    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE)

    # Plot ROC curve
    ax.plot(
        fpr, tpr,
        color="steelblue",
        linewidth=2,
        label=f"Logistic Regression (AUC = {auc_score:.4f})"
    )

    # Plot random baseline
    ax.plot(
        [0, 1], [0, 1],
        color="gray",
        linewidth=1,
        linestyle="--",
        label="Random Classifier (AUC = 0.50)"
    )

    # Highlight the point closest to perfect classification
    optimal_idx = np.argmax(tpr - fpr)
    ax.scatter(
        fpr[optimal_idx], tpr[optimal_idx],
        color="red", s=100, zorder=5,
        label=f"Optimal threshold = {thresholds[optimal_idx]:.2f}"
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate (Recall)", fontsize=12)
    ax.set_title("ROC AUC Curve — Telecom Churn Prediction", fontsize=14)
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)

    save_figure(fig, "roc_curve.png")

    return auc_score


# =============================================================================
# PRECISION RECALL CURVE
# =============================================================================

def plot_precision_recall_curve(y_test, y_prob) -> float:
    """
    Plot and save the precision recall curve.

    More informative than ROC for imbalanced datasets.
    Shows the tradeoff between precision and recall across thresholds.
    The business uses this to choose the operating threshold based
    on the cost of false positives vs false negatives.

    Parameters
    ----------
    y_test : pd.Series
        True labels.
    y_prob : np.ndarray
        Predicted churn probabilities.

    Returns
    -------
    float
        Average precision score (area under precision recall curve).
    """
    logger.info("Plotting precision recall curve")

    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)

    # Baseline is the churn rate in the test set
    baseline = y_test.mean()

    logger.info(f"Average Precision Score: {avg_precision:.4f}")
    logger.info(f"Baseline (churn rate)  : {baseline:.4f}")

    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE)

    ax.plot(
        recall, precision,
        color="steelblue",
        linewidth=2,
        label=f"Logistic Regression (AP = {avg_precision:.4f})"
    )

    # Baseline — a random classifier would achieve precision = churn rate
    ax.axhline(
        y=baseline,
        color="gray",
        linewidth=1,
        linestyle="--",
        label=f"Random Classifier (AP = {baseline:.4f})"
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title("Precision Recall Curve — Telecom Churn Prediction", fontsize=14)
    ax.legend(loc="upper right", fontsize=10)
    ax.grid(True, alpha=0.3)

    save_figure(fig, "precision_recall_curve.png")

    return avg_precision


# =============================================================================
# SHAP ANALYSIS
# =============================================================================

def get_shap_values(model, X_test_processed: np.ndarray):
    """
    Calculate SHAP values for the test set.

    Uses LinearExplainer which is optimised for linear models like
    logistic regression. It is much faster than the general
    KernelExplainer for this use case.

    Parameters
    ----------
    model : fitted Pipeline
        Trained model pipeline containing logistic regression.
    X_test_processed : np.ndarray
        Processed test features — same array used for predictions.

    Returns
    -------
    shap_values : np.ndarray
        SHAP values for each feature and each customer.
        Shape: (n_customers, n_features)
    explainer : shap.LinearExplainer
        Fitted SHAP explainer object.
    """
    logger.info("Calculating SHAP values")

    # Extract the logistic regression step from the pipeline
    lr_model = model.named_steps["logisticregression"]

    # LinearExplainer is optimised for linear models
    explainer = shap.LinearExplainer(
        lr_model,
        X_test_processed,
        feature_perturbation="interventional"
    )

    shap_values = explainer.shap_values(X_test_processed)

    logger.info(f"SHAP values calculated: {shap_values.shape}")

    return shap_values, explainer


def get_feature_names(pipeline, preprocessing_pipeline) -> list:
    """
    Extract feature names from the preprocessing pipeline.

    After OneHotEncoding the feature names become complex.
    This function retrieves the exact names in the correct order
    so SHAP plots have meaningful labels.

    Parameters
    ----------
    pipeline : fitted ImbPipeline
        Trained model pipeline.
    preprocessing_pipeline : fitted Pipeline
        Preprocessing pipeline with ColumnTransformer.

    Returns
    -------
    list
        List of feature names in the same order as the processed array.
    """
    logger.info("Extracting feature names from preprocessing pipeline")

    try:
        feature_names = (
            preprocessing_pipeline
            .named_steps["preprocessor"]
            .get_feature_names_out()
            .tolist()
        )
        logger.info(f"Retrieved {len(feature_names)} feature names")
        return feature_names
    except Exception as e:
        logger.warning(f"Could not retrieve feature names: {e}")
        logger.warning("Using generic feature names instead")
        return [f"feature_{i}" for i in range(40)]


def plot_shap_summary(shap_values, X_test_processed, feature_names) -> None:
    """
    Plot and save the SHAP summary plot.

    Shows all features ranked by their average absolute SHAP value.
    Each dot represents one customer coloured by the feature value.
    Red = high feature value, Blue = low feature value.

    Parameters
    ----------
    shap_values : np.ndarray
        SHAP values for all customers and features.
    X_test_processed : np.ndarray
        Processed test features.
    feature_names : list
        Names of all features.

    Returns
    -------
    None
    """
    logger.info("Plotting SHAP summary plot")

    fig, ax = plt.subplots(figsize=(10, 8))

    shap.summary_plot(
        shap_values,
        X_test_processed,
        feature_names=feature_names,
        show=False,
        max_display=20,      # Show top 20 most important features
        plot_size=None
    )

    plt.title("SHAP Summary Plot — Feature Importance", fontsize=14, pad=15)
    plt.tight_layout()

    save_figure(plt.gcf(), "shap_summary.png")
    logger.info("SHAP summary plot saved")


def plot_shap_waterfall(shap_values, X_test_processed,
                        feature_names, explainer,
                        customer_index: int = 0) -> None:
    """
    Plot and save the SHAP waterfall plot for one specific customer.

    Shows exactly how each feature pushed the prediction up or down
    from the baseline expected value. This is the local explanation
    used by the retention team to understand why a specific customer
    was flagged.

    Parameters
    ----------
    shap_values : np.ndarray
        SHAP values for all customers.
    X_test_processed : np.ndarray
        Processed test features.
    feature_names : list
        Names of all features.
    explainer : shap.LinearExplainer
        Fitted SHAP explainer.
    customer_index : int
        Index of the customer to explain. Default is 0 (first customer).

    Returns
    -------
    None
    """
    logger.info(f"Plotting SHAP waterfall plot for customer index {customer_index}")

    # Create SHAP Explanation object for the waterfall plot
    explanation = shap.Explanation(
        values=shap_values[customer_index],
        base_values=explainer.expected_value,
        data=X_test_processed[customer_index],
        feature_names=feature_names
    )

    fig, ax = plt.subplots(figsize=(10, 8))

    shap.waterfall_plot(
        explanation,
        max_display=15,
        show=False
    )

    plt.title(
        f"SHAP Waterfall Plot — Customer {customer_index}",
        fontsize=14, pad=15
    )
    plt.tight_layout()

    save_figure(plt.gcf(), f"shap_waterfall_customer_{customer_index}.png")
    logger.info("SHAP waterfall plot saved")


def plot_shap_dependence(shap_values, X_test_processed,
                         feature_names) -> None:
    """
    Plot and save SHAP dependence plots for the top 3 features.

    Shows how the effect of one feature on churn probability
    changes across its range. Reveals non-linear relationships
    and interaction effects.

    Parameters
    ----------
    shap_values : np.ndarray
        SHAP values for all customers.
    X_test_processed : np.ndarray
        Processed test features.
    feature_names : list
        Names of all features.

    Returns
    -------
    None
    """
    logger.info("Plotting SHAP dependence plots for top 3 features")

    # Find top 3 features by mean absolute SHAP value
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top3_indices = np.argsort(mean_abs_shap)[::-1][:3]
    top3_features = [feature_names[i] for i in top3_indices]

    logger.info(f"Top 3 features: {top3_features}")

    for i, feature in enumerate(top3_features):
        fig, ax = plt.subplots(figsize=config.FIGURE_SIZE)

        shap.dependence_plot(
            feature,
            shap_values,
            X_test_processed,
            feature_names=feature_names,
            ax=ax,
            show=False
        )

        ax.set_title(
            f"SHAP Dependence Plot — {feature}",
            fontsize=14, pad=15
        )
        plt.tight_layout()

        save_figure(fig, f"shap_dependence_{feature.replace(' ', '_')}.png")

    logger.info("SHAP dependence plots saved")


# =============================================================================
# SAVE RESULTS
# =============================================================================

def save_results(metrics: dict) -> None:
    """
    Save all evaluation metrics to a JSON file for future reference.

    Having results in a file means we never lose them between sessions.
    They can also be read programmatically for reporting.

    Parameters
    ----------
    metrics : dict
        Dictionary containing all evaluation metrics.

    Returns
    -------
    None
    """
    os.makedirs(config.REPORTS_DIR, exist_ok=True)
    results_path = os.path.join(config.REPORTS_DIR, "evaluation_results.json")

    with open(results_path, "w") as f:
        json.dump(metrics, f, indent=4)

    logger.info(f"Evaluation results saved to: {results_path}")


# =============================================================================
# MASTER EVALUATION FUNCTION
# =============================================================================

def run_evaluation_pipeline(
    model,
    X_test_processed: np.ndarray,
    y_test,
    preprocessing_pipeline,
    customer_index: int = 0
) -> dict:
    """
    Master function that runs the complete evaluation pipeline.

    This is the main function called by main.py.

    Steps:
        1. Generate predictions
        2. Plot confusion matrix
        3. Log classification report
        4. Plot ROC AUC curve
        5. Plot precision recall curve
        6. Calculate SHAP values
        7. Plot SHAP summary
        8. Plot SHAP waterfall for one customer
        9. Plot SHAP dependence plots for top 3 features
        10. Save all metrics to JSON

    Parameters
    ----------
    model : fitted Pipeline
        Trained model pipeline.
    X_test_processed : np.ndarray
        Processed test features.
    y_test : pd.Series
        True test labels.
    preprocessing_pipeline : Pipeline
        Fitted preprocessing pipeline for feature name extraction.
    customer_index : int
        Customer index to use for waterfall plot. Default 0.

    Returns
    -------
    dict
        All evaluation metrics in one dictionary.
    """
    logger.info("=" * 60)
    logger.info("STARTING EVALUATION PIPELINE")
    logger.info("=" * 60)

    # Step 1: Generate predictions
    y_pred, y_prob = get_predictions(model, X_test_processed)

    # Step 2: Confusion matrix
    cm_results = plot_confusion_matrix(y_test, y_pred)

    # Step 3: Classification report
    report_dict = log_classification_report(y_test, y_pred)

    # Step 4: ROC AUC curve
    auc_score = plot_roc_curve(y_test, y_prob)

    # Step 5: Precision recall curve
    avg_precision = plot_precision_recall_curve(y_test, y_prob)

    # Step 6: Get feature names
    feature_names = get_feature_names(model, preprocessing_pipeline)

    # Step 7: Calculate SHAP values
    shap_values, explainer = get_shap_values(model, X_test_processed)

    # Step 8: SHAP summary plot
    plot_shap_summary(shap_values, X_test_processed, feature_names)

    # Step 9: SHAP waterfall plot
    plot_shap_waterfall(
        shap_values, X_test_processed,
        feature_names, explainer,
        customer_index
    )

    # Step 10: SHAP dependence plots
    plot_shap_dependence(shap_values, X_test_processed, feature_names)

    # Compile all metrics
    metrics = {
        "roc_auc_score"       : round(auc_score, 4),
        "average_precision"   : round(avg_precision, 4),
        "confusion_matrix"    : cm_results,
        "classification_report": report_dict
    }

    # Save results to JSON
    save_results(metrics)

    logger.info("=" * 60)
    logger.info("EVALUATION PIPELINE COMPLETE")
    logger.info(f"ROC AUC Score     : {auc_score:.4f}")
    logger.info(f"Average Precision : {avg_precision:.4f}")
    logger.info("All plots saved to reports/figures/")
    logger.info("=" * 60)

    return metrics
'''

# Write evaluate.py to src folder
evaluate_path = os.path.join(PROJECT_PATH, 'src', 'evaluate.py')

with open(evaluate_path, 'w') as f:
    f.write(evaluate_content)

print(f"evaluate.py written successfully to:")
print(f"  {evaluate_path}")

evaluate.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/evaluate.py


In [42]:

!pip install shap --quiet
print("shap installed successfully!")

shap installed successfully!


In [43]:
#Verify if Evaluation works.
import importlib
import evaluate
importlib.reload(evaluate)

# Run full evaluation pipeline
metrics = evaluate.run_evaluation_pipeline(
    model=best_model,
    X_test_processed=X_test_processed,
    y_test=y_test,
    preprocessing_pipeline=pipeline,
    customer_index=0
)

# Show final summary
print("\nFinal Evaluation Summary:")
print(f"  ROC AUC Score     : {metrics['roc_auc_score']}")
print(f"  Average Precision : {metrics['average_precision']}")
print(f"  True Positives    : {metrics['confusion_matrix']['TP']}")
print(f"  False Negatives   : {metrics['confusion_matrix']['FN']}")


CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Stayed (0)       0.89      0.75      0.82      1033
 Churned (1)       0.52      0.75      0.62       372

    accuracy                           0.75      1405
   macro avg       0.71      0.75      0.72      1405
weighted avg       0.80      0.75      0.76      1405



/usr/local/lib/python3.12/dist-packages/shap/explainers/_linear.py:123: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)



Final Evaluation Summary:
  ROC AUC Score     : 0.8413
  Average Precision : 0.6502
  True Positives    : 280
  False Negatives   : 92


In [44]:
# Verify that all plots were saved
# Check all plots exist in reports/figures
figures_dir = os.path.join(PROJECT_PATH, 'reports', 'figures')

print("Saved figures:")
for f in sorted(os.listdir(figures_dir)):
    size_kb = os.path.getsize(os.path.join(figures_dir, f)) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

Saved figures:
  confusion_matrix.png (148.7 KB)
  precision_recall_curve.png (148.9 KB)
  roc_curve.png (191.8 KB)
  shap_dependence_ContractRiskScore.png (123.5 KB)
  shap_dependence_InternetService_Fiber_optic.png (135.5 KB)
  shap_dependence_TenureContractInteraction.png (167.5 KB)
  shap_summary.png (387.7 KB)
  shap_waterfall_customer_0.png (322.3 KB)


In [7]:

# =============================================================================
# CHAPTER 8: PREDICTION
# Writes predict.py to src/
# Only run once — file stays in Drive permanently
# =============================================================================

In [8]:

predict_content = '''
# =============================================================================
# predict.py
# Handles all prediction logic for the trained churn model.
# Supports single customer prediction, batch prediction,
# and prediction explanation for the retention team.
# =============================================================================

import logging
import numpy as np
import pandas as pd
import os
import sys
import joblib

sys.path.append(os.path.dirname(os.path.abspath(__file__)))
import config

logging.basicConfig(level=config.LOG_LEVEL, format=config.LOG_FORMAT)
logger = logging.getLogger(__name__)


# =============================================================================
# RISK CATEGORY DEFINITIONS
# =============================================================================

RISK_CATEGORIES = [
    (0.00, 0.30, "Low Risk",      "No action needed — customer is stable"),
    (0.30, 0.50, "Medium Risk",   "Monitor closely — schedule check-in call"),
    (0.50, 0.70, "High Risk",     "Proactive outreach — offer loyalty discount"),
    (0.70, 1.00, "Critical Risk", "Immediate intervention — escalate to retention team")
]


def load_model(model_path: str = config.MODEL_PATH):
    """
    Load the saved model pipeline from disk.

    Parameters
    ----------
    model_path : str
        Path to the saved .pkl model file.

    Returns
    -------
    pipeline : fitted Pipeline
        Complete end-to-end prediction pipeline.

    Raises
    ------
    FileNotFoundError
        If no saved model exists at the given path.
    """
    logger.info(f"Loading model from: {model_path}")

    if not os.path.exists(model_path):
        logger.error(f"No model found at: {model_path}")
        raise FileNotFoundError(
            f"No saved model found at {model_path}. "
            f"Please run train.py first."
        )

    pipeline = joblib.load(model_path)
    logger.info(f"Model loaded successfully: {type(pipeline).__name__}")

    return pipeline


def get_risk_category(probability: float) -> tuple:
    """
    Convert a churn probability into a business friendly risk category.

    Parameters
    ----------
    probability : float
        Churn probability between 0 and 1.

    Returns
    -------
    tuple
        (risk_category, recommendation) strings.
    """
    for low, high, category, recommendation in RISK_CATEGORIES:
        if low <= probability < high:
            return category, recommendation

    # Handle edge case of exactly 1.0
    return "Critical Risk", "Immediate intervention — escalate to retention team"


def validate_input(customer_data: dict) -> None:
    """
    Validate that input customer data contains all required columns.

    Parameters
    ----------
    customer_data : dict
        Raw customer data as a dictionary.

    Raises
    ------
    ValueError
        If any required columns are missing from the input.
    """
    required_columns = (
        config.NUMERICAL_FEATURES +
        config.BINARY_FEATURES +
        config.MULTICLASS_FEATURES +
        [config.SENIOR_CITIZEN_FEATURE]
    )

    missing = [col for col in required_columns if col not in customer_data]

    if missing:
        logger.error(f"Missing required fields: {missing}")
        raise ValueError(
            f"Customer data is missing required fields: {missing}"
        )

    logger.info("Input validation passed")


def predict_single_customer(
    customer_data: dict,
    model,
    threshold: float = config.CLASSIFICATION_THRESHOLD
) -> dict:
    """
    Predict churn probability for a single customer.

    Takes raw customer data exactly as it comes from the business
    system — no preprocessing required. The pipeline handles
    all feature engineering and preprocessing internally.

    Parameters
    ----------
    customer_data : dict
        Raw customer data as key-value pairs.
        Must contain all required fields.
    model : fitted Pipeline
        Loaded model pipeline.
    threshold : float
        Classification threshold. Default 0.5.
        Lower threshold catches more churners but increases false alarms.

    Returns
    -------
    dict
        Complete prediction result including:
        - churn_probability
        - churn_predicted
        - risk_category
        - recommendation
    """
    logger.info("Predicting churn for single customer")

    # Validate input
    validate_input(customer_data)

    # Convert dict to dataframe — pipeline expects a dataframe
    customer_df = pd.DataFrame([customer_data])

    # Get churn probability
    # predict_proba returns [[prob_stay, prob_churn]]
    # [:, 1] selects prob_churn
    churn_probability = model.predict_proba(customer_df)[:, 1][0]

    # Apply threshold to get binary prediction
    churn_predicted = int(churn_probability >= threshold)

    # Get risk category and recommendation
    risk_category, recommendation = get_risk_category(churn_probability)

    result = {
        "churn_probability" : round(float(churn_probability), 4),
        "churn_predicted"   : churn_predicted,
        "risk_category"     : risk_category,
        "recommendation"    : recommendation,
        "threshold_used"    : threshold
    }

    logger.info(f"Prediction complete:")
    logger.info(f"  Churn probability : {result['churn_probability']:.4f}")
    logger.info(f"  Risk category     : {result['risk_category']}")
    logger.info(f"  Recommendation    : {result['recommendation']}")

    return result


def predict_batch(
    customers_df: pd.DataFrame,
    model,
    threshold: float = config.CLASSIFICATION_THRESHOLD
) -> pd.DataFrame:
    """
    Predict churn probability for a batch of customers.

    Parameters
    ----------
    customers_df : pd.DataFrame
        Raw customer data. Each row is one customer.
        Must contain all required columns.
    model : fitted Pipeline
        Loaded model pipeline.
    threshold : float
        Classification threshold. Default 0.5.

    Returns
    -------
    pd.DataFrame
        Original dataframe with four new columns appended:
        - ChurnProbability
        - ChurnPredicted
        - RiskCategory
        - Recommendation
    """
    logger.info(f"Predicting churn for batch of {len(customers_df)} customers")

    # Work on a copy to avoid modifying the original
    results_df = customers_df.copy()

    # Get probabilities for all customers at once
    churn_probabilities = model.predict_proba(customers_df)[:, 1]

    # Apply threshold
    churn_predicted = (churn_probabilities >= threshold).astype(int)

    # Get risk categories for all customers
    risk_categories = []
    recommendations = []

    for prob in churn_probabilities:
        category, recommendation = get_risk_category(prob)
        risk_categories.append(category)
        recommendations.append(recommendation)

    # Append results to dataframe
    results_df["ChurnProbability"] = np.round(churn_probabilities, 4)
    results_df["ChurnPredicted"]   = churn_predicted
    results_df["RiskCategory"]     = risk_categories
    results_df["Recommendation"]   = recommendations

    # Log summary statistics
    logger.info(f"Batch prediction complete:")
    logger.info(f"  Total customers    : {len(results_df)}")
    logger.info(f"  Predicted churners : {churn_predicted.sum()}")
    logger.info(f"  Predicted churn rate: {churn_predicted.mean():.2%}")

    logger.info("Risk category distribution:")
    for category in ["Low Risk", "Medium Risk", "High Risk", "Critical Risk"]:
        count = (results_df["RiskCategory"] == category).sum()
        pct = count / len(results_df) * 100
        logger.info(f"  {category:<15}: {count} customers ({pct:.1f}%)")

    return results_df


def run_prediction_pipeline(
    model,
    X_test_raw: pd.DataFrame,
    y_test: pd.Series
) -> pd.DataFrame:
    """
    Master function that runs batch prediction on the test set
    and produces a full results dataframe for analysis.

    This is the main function called by main.py.

    Parameters
    ----------
    model : fitted Pipeline
        Loaded model pipeline.
    X_test_raw : pd.DataFrame
        Raw unprocessed test features.
        Note: We use raw features here because the pipeline
        handles all preprocessing internally.
    y_test : pd.Series
        True test labels for comparison.

    Returns
    -------
    pd.DataFrame
        Complete results dataframe with predictions and actual labels.
    """
    logger.info("=" * 60)
    logger.info("STARTING PREDICTION PIPELINE")
    logger.info("=" * 60)

    # Run batch prediction
    results_df = predict_batch(X_test_raw, model)

    # Add actual labels for comparison
    results_df["ActualChurn"] = y_test.values

    # Add correct/incorrect flag for easy filtering
    results_df["CorrectPrediction"] = (
        results_df["ChurnPredicted"] == results_df["ActualChurn"]
    ).astype(int)

    # Save results to CSV for business use
    results_path = os.path.join(config.REPORTS_DIR, "predictions.csv")
    os.makedirs(config.REPORTS_DIR, exist_ok=True)
    results_df.to_csv(results_path, index=False)
    logger.info(f"Predictions saved to: {results_path}")

    logger.info("=" * 60)
    logger.info("PREDICTION PIPELINE COMPLETE")
    logger.info("=" * 60)

    return results_df
'''
# Write predict.py to src folder
predict_path = os.path.join(PROJECT_PATH, 'src', 'predict.py')

with open(predict_path, 'w') as f:
    f.write(predict_content)

print(f"predict.py written successfully to:")
print(f"  {predict_path}")

predict.py written successfully to:
  /content/drive/MyDrive/telecom_churn_prediction/src/predict.py


In [9]:
# Verify if prediction works
import importlib
import predict
importlib.reload(predict)

# Test 1: Single customer prediction
# This is a high risk customer — new, month-to-month, fiber optic
high_risk_customer = {
    "gender"           : "Female",
    "SeniorCitizen"    : 0,
    "Partner"          : "No",
    "Dependents"       : "No",
    "tenure"           : 2,
    "PhoneService"     : "Yes",
    "MultipleLines"    : "No",
    "InternetService"  : "Fiber optic",
    "OnlineSecurity"   : "No",
    "OnlineBackup"     : "No",
    "DeviceProtection" : "No",
    "TechSupport"      : "No",
    "StreamingTV"      : "Yes",
    "StreamingMovies"  : "Yes",
    "Contract"         : "Month-to-month",
    "PaperlessBilling" : "Yes",
    "PaymentMethod"    : "Electronic check",
    "MonthlyCharges"   : 85.50,
    "TotalCharges"     : 171.00
}

print("=" * 60)
print("TEST 1: HIGH RISK CUSTOMER")
print("=" * 60)
result = predict.predict_single_customer(high_risk_customer, best_model)
for key, value in result.items():
    print(f"  {key:<20}: {value}")

# Test 2: Single customer prediction
# This is a low risk customer — long tenure, two year contract
low_risk_customer = {
    "gender"           : "Male",
    "SeniorCitizen"    : 0,
    "Partner"          : "Yes",
    "Dependents"       : "Yes",
    "tenure"           : 60,
    "PhoneService"     : "Yes",
    "MultipleLines"    : "Yes",
    "InternetService"  : "DSL",
    "OnlineSecurity"   : "Yes",
    "OnlineBackup"     : "Yes",
    "DeviceProtection" : "Yes",
    "TechSupport"      : "Yes",
    "StreamingTV"      : "No",
    "StreamingMovies"  : "No",
    "Contract"         : "Two year",
    "PaperlessBilling" : "No",
    "PaymentMethod"    : "Bank transfer (automatic)",
    "MonthlyCharges"   : 65.00,
    "TotalCharges"     : 3900.00
}

print("\n" + "=" * 60)
print("TEST 2: LOW RISK CUSTOMER")
print("=" * 60)
result = predict.predict_single_customer(low_risk_customer, best_model)
for key, value in result.items():
    print(f"  {key:<20}: {value}")

# Test 3: Batch prediction on test set
print("\n" + "=" * 60)
print("TEST 3: BATCH PREDICTION ON TEST SET")
print("=" * 60)

# Get raw test features
X, y = preprocessing.split_features_target(df_clean)
X_train, X_test_raw, y_train, y_test = preprocessing.split_train_test(X, y)

results_df = predict.run_prediction_pipeline(best_model, X_test_raw, y_test)

print(f"\nResults dataframe shape: {results_df.shape}")
print(f"\nRisk category distribution:")
print(results_df["RiskCategory"].value_counts())
print(f"\nFirst 5 predictions:")
display(results_df[[
    "tenure", "Contract", "MonthlyCharges",
    "ChurnProbability", "RiskCategory", "ActualChurn"
]].head())

TEST 1: HIGH RISK CUSTOMER


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


ValueError: could not convert string to float: 'Female'

Creating a combined pipeline that chains preprocessing and the model together

In [11]:

# Combine preprocessing pipeline and best model into one
# end to end pipeline that takes raw data and returns predictions

from sklearn.pipeline import Pipeline as SklearnPipeline
import joblib

# Extract just the logistic regression from best_model
# We do not include SMOTE in the prediction pipeline
# SMOTE is only needed during training not prediction
lr_model = best_model.named_steps['logisticregression']

# Build complete end to end pipeline
full_prediction_pipeline = SklearnPipeline(steps=[
    ('preprocessing', pipeline),        # feature engineering + encoding + scaling
    ('logisticregression', lr_model)    # trained logistic regression
])

# Save this complete pipeline to disk
full_pipeline_path = os.path.join(PROJECT_PATH, 'models', 'full_prediction_pipeline.pkl')
joblib.dump(full_prediction_pipeline, full_pipeline_path)

print("Full prediction pipeline saved!")
print(f"Location: {full_pipeline_path}")
print(f"\nPipeline steps:")
for step_name, step in full_prediction_pipeline.steps:
    print(f"  {step_name}: {type(step).__name__}")

Full prediction pipeline saved!
Location: /content/drive/MyDrive/telecom_churn_prediction/models/full_prediction_pipeline.pkl

Pipeline steps:
  preprocessing: Pipeline
  logisticregression: LogisticRegression


In [17]:
#Update config.py to Add the New Path
# Add full pipeline path to config
config_path = os.path.join(PROJECT_PATH, 'src', 'config.py')

with open(config_path, 'r') as f:
    config_content = f.read()

# Add new path after MODEL_PATH line
config_content = config_content.replace(
    "MODEL_PATH = os.path.join(MODEL_DIR, 'logistic_regression_pipeline.pkl')",
    "MODEL_PATH = os.path.join(MODEL_DIR, 'logistic_regression_pipeline.pkl')\n\n# Complete end to end pipeline for production predictions\nFULL_PIPELINE_PATH = os.path.join(MODEL_DIR, 'full_prediction_pipeline.pkl')"
)

with open(config_path, 'w') as f:
    f.write(config_content)

# Reload config
importlib.reload(config)
print("config.py updated with FULL_PIPELINE_PATH")
print(f"FULL_PIPELINE_PATH: {config.FULL_PIPELINE_PATH}")

config.py updated with FULL_PIPELINE_PATH
FULL_PIPELINE_PATH: /content/drive/MyDrive/telecom_churn_prediction/models/full_prediction_pipeline.pkl


In [18]:
#Rerun the Verification Using the Full Pipeline
importlib.reload(predict)

# Load the full prediction pipeline
full_pipeline = joblib.load(config.FULL_PIPELINE_PATH)

print("Full pipeline steps:")
for step_name, step in full_pipeline.steps:
    print(f"  {step_name}: {type(step).__name__}")

# Test 1: High risk customer
print("\n" + "=" * 60)
print("TEST 1: HIGH RISK CUSTOMER")
print("=" * 60)
result = predict.predict_single_customer(high_risk_customer, full_pipeline)
for key, value in result.items():
    print(f"  {key:<20}: {value}")

# Test 2: Low risk customer
print("\n" + "=" * 60)
print("TEST 2: LOW RISK CUSTOMER")
print("=" * 60)
result = predict.predict_single_customer(low_risk_customer, full_pipeline)
for key, value in result.items():
    print(f"  {key:<20}: {value}")

# Test 3: Batch prediction
print("\n" + "=" * 60)
print("TEST 3: BATCH PREDICTION ON TEST SET")
print("=" * 60)

X, y = preprocessing.split_features_target(df_clean)
X_train, X_test_raw, y_train_raw, y_test_raw = preprocessing.split_train_test(X, y)

results_df = predict.run_prediction_pipeline(full_pipeline, X_test_raw, y_test_raw)

print(f"\nResults dataframe shape: {results_df.shape}")
print(f"\nRisk category distribution:")
print(results_df["RiskCategory"].value_counts())
print(f"\nFirst 5 predictions:")
display(results_df[[
    "tenure", "Contract", "MonthlyCharges",
    "ChurnProbability", "RiskCategory", "ActualChurn"
]].head())

Full pipeline steps:
  preprocessing: Pipeline
  logisticregression: LogisticRegression

TEST 1: HIGH RISK CUSTOMER
  churn_probability   : 0.9374
  churn_predicted     : 1
  risk_category       : Critical Risk
  recommendation      : Immediate intervention — escalate to retention team
  threshold_used      : 0.5

TEST 2: LOW RISK CUSTOMER
  churn_probability   : 0.0219
  churn_predicted     : 0
  risk_category       : Low Risk
  recommendation      : No action needed — customer is stable
  threshold_used      : 0.5

TEST 3: BATCH PREDICTION ON TEST SET

Results dataframe shape: (1405, 25)

Risk category distribution:
RiskCategory
Low Risk         615
Critical Risk    322
Medium Risk      255
High Risk        213
Name: count, dtype: int64

First 5 predictions:


,tenure,Contract,MonthlyCharges,ChurnProbability,RiskCategory,ActualChurn
5627,7,Month-to-month,74.85,0.9099,Critical Risk,0
6126,19,Month-to-month,95.90,0.7577,Critical Risk,1
2361,1,Month-to-month,45.95,0.5311,High Risk,1
2201,36,One year,85.85,0.1110,Low Risk,0
832,70,One year,74.10,0.0333,Low Risk,0


In [16]:

# Define both test customers
high_risk_customer = {
    "gender"           : "Female",
    "SeniorCitizen"    : 0,
    "Partner"          : "No",
    "Dependents"       : "No",
    "tenure"           : 2,
    "PhoneService"     : "Yes",
    "MultipleLines"    : "No",
    "InternetService"  : "Fiber optic",
    "OnlineSecurity"   : "No",
    "OnlineBackup"     : "No",
    "DeviceProtection" : "No",
    "TechSupport"      : "No",
    "StreamingTV"      : "Yes",
    "StreamingMovies"  : "Yes",
    "Contract"         : "Month-to-month",
    "PaperlessBilling" : "Yes",
    "PaymentMethod"    : "Electronic check",
    "MonthlyCharges"   : 85.50,
    "TotalCharges"     : 171.00
}

low_risk_customer = {
    "gender"           : "Male",
    "SeniorCitizen"    : 0,
    "Partner"          : "Yes",
    "Dependents"       : "Yes",
    "tenure"           : 60,
    "PhoneService"     : "Yes",
    "MultipleLines"    : "Yes",
    "InternetService"  : "DSL",
    "OnlineSecurity"   : "Yes",
    "OnlineBackup"     : "Yes",
    "DeviceProtection" : "Yes",
    "TechSupport"      : "Yes",
    "StreamingTV"      : "No",
    "StreamingMovies"  : "No",
    "Contract"         : "Two year",
    "PaperlessBilling" : "No",
    "PaymentMethod"    : "Bank transfer (automatic)",
    "MonthlyCharges"   : 65.00,
    "TotalCharges"     : 3900.00
}

print("Both customers defined successfully!")

Both customers defined successfully!
